## 📑 HAPS Collections Report
 
#### 🔗 Links
- **[BRD link](https://cepalhellas.atlassian.net/wiki/x/IIAwH)**  
  Comprehensive documentation outlining the business requirements for this report.

##### Authentication

In [0]:
%python
import boto3
from botocore.exceptions import NoCredentialsError
import traceback

"""
Accessing the Hive metastore requires specific considerations:

1. Hive metastore access is not possible via the instance profile if the cluster is on shared access mode.
   The cluster must be in shared access mode to interact with the Unity catalog.

2. To access both Unity catalog and AWS services, an access token from Amazon is required.

3. To retrieve the token secret key, access the secrets scope 'aws-databricks,' where the access key is stored per environment.
    https://docs.databricks.com/en/security/secrets/index.html

4. Once AWS credentials are obtained, clients for services like S3 and SSM can be created.

5. After you authenticate successfully querying the Hive metastore is also working.
"""

try:
    mode = dbutils.widgets.get("mode")
except Exception as e:
    mode = "shared" 

# Function to get AWS credentials from Databricks secrets.
def get_aws_credentials(secrets_scope, key_name):
    try:
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None
    
# Function to get AWS credentials from Databricks secrets.
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration.
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
    
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

secrets_scope = "aws-databricks"
aws_keys = get_scope_key_names(secrets_scope)

aws_key = aws_keys[0] if aws_keys else None

# Get AWS secret key from Databricks secrets.
aws_secret_key = get_aws_credentials(secrets_scope,aws_key)

if mode == "shared":
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)

    S3_client = boto3.client('s3', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)                           
else:
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1')

    S3_client = boto3.client('s3', region_name='eu-central-1')                           


# Function to connect to AWS and access SSM parameter.
def connect_to_aws_and_access_ssm(ssm_key):

    if aws_secret_key is None:
        print("AWS secret key not available.")
        return

    # Access SSM parameter /cfn/a360/reports_output_path.
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )

    return response['Parameter']['Value']

# Get the Buckewt prefix from SSM to be sure.
env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix')

In [0]:
%python
from pyspark.sql.functions import coalesce, concat_ws, lit, sha2, substring, when, col, regexp_replace

import delta_table_creator, extract, transform
from delta import DeltaTable  # type: ignore
import boto3
import os

dbutils.widgets.text("run_date", '')
run_date = dbutils.widgets.get("run_date").strip()

if not run_date:
    run_date = spark.sql(f'''
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.derivative.dexposure 
                         union
                         select MAX(BusinessDate) BusinessDate from `p-cdh`.landed.profits_loan_details
                         ''').collect()

    if len(run_date) != 1: raise ValueError(f"Incorrect run_date!")
    run_date = run_date[0]['BusinessDate']
    run_date = run_date.strftime('%Y-%m-%d') 

# from pyspark.sql import SparkSession
# spark = SparkSession.builder.appName("example").getOrCreate()

from pyspark.sql import DataFrame
from pyspark.sql.functions import trim

In [0]:
%python
table_name = "HAPS_CollectionsReport"

schema = (
    "SoMBusinessDate                      DATE         ,"    
    "BusinessDate                         DATE         ," 
    "ExposureKey                          STRING       ,"
    "CDI                                   STRING       ,"
    "AgreementKey                          STRING       ,"
    "PortfolioName                         STRING       ,"
    "SoMIsActiveFlag                       INT          ,"
    "AccountBalanceFlag                    STRING       ,"
    "AccountType                           STRING       ,"
    "GAccountFlag                          STRING       ,"
    "AssetSegment                          STRING       ,"
    "SubProductDescription                 STRING       ,"
    "ProductDescription                    STRING       ,"
    "SoMBucket                             INT          ,"
    "SoMBucketGroup                        STRING       ,"
    "SoMArrearsStatusEng                   STRING       ,"
    "MtDMBucket                            INT          ,"
    "MtDBucketGroup                        STRING       ,"
    "ObjectiveStatus                       STRING       ,"
    "ObjectiveDate                         DATE         ,"
    "CustomerPayerSegment                  STRING       ,"
    "AccountPayerSegment                   STRING       ,"
    "FinalSegment                          STRING       ,"
    "OfficialSegment                       STRING       ,"
    "L3869Flag                             STRING       ,"
    "SoML3869Flag                          STRING       ,"
    "SoML3869Status                        STRING       ,"
    "MasterAccount                         STRING       ,"
    "FullyWoff                             INT          ,"
    "FullyClosed                           INT          ,"
    "HasActiveSettlementFlag               INT          ,"
    "NextInstallmentDate                   DATE         ,"
    "NextInstallmentAmount                 DECIMAL(19,2),"
    "SoMTotalLegalClaimEur                 DECIMAL(19,2),"
    "SoMGBVEur                              DECIMAL(19,2),"
    "onbalanceamount                       DECIMAL(19,2),"
    "offbalanceamount                      DECIMAL(19,2),"
    "SoMDelinquent                         DECIMAL(19,2),"
    "EoMDelinquentAmount                   DECIMAL(19,2),"
    "TargetRate                            DECIMAL(19,2),"
    "TargetAmount                          DECIMAL(19,2),"
    "RemainBalance                         DECIMAL(19,2),"
    "RollForwardBalance                    DECIMAL(19,2),"
    "RollBackBalance                       DECIMAL(19,2),"
    "ContractLevelAssignmentCategory       STRING       ,"
    "ReportingCategoryLevel3               STRING       ,"
    "ReportingCategoryLevel2               STRING       ,"
    "ReportingCategoryLevel1               STRING       ,"
    "CashTransactions                      DECIMAL(19,2),"
    "LiquidationTransactions               DECIMAL(19,2),"
    "CashPayments                          DECIMAL(19,2),"
    "Liquidations                          DECIMAL(19,2),"
    "CashTransactionsRestricted            DECIMAL(19,2),"
    "CashPaymentsRestricted                DECIMAL(19,2),"
    "QueueDescription                      STRING       ,"
    "AssignmentChannel                     STRING       ,"
    "AssignmentTeam                        STRING       ,"
    "AssignmentAgent                       STRING       ,"
    "IntramonthFlag                        INT          ,"
    "ApplicationImplementationStatus       STRING       ,"
    "Exceptions                            STRING       ,"
    "DownPayment                           DECIMAL(19,2),"
    "DPOPayment                            DECIMAL(19,2),"
    "ExpenseRequestAmount                  DECIMAL(19,2),"
    "PaymentJumpDate                       DATE         ,"
    "PaymentJumpAmount                     DECIMAL(19,2),"
    "HighBalanceFlag                       STRING       ,"
    "InsertTimeStamp                       TIMESTAMP"
)


In [0]:
%python
partition_columns = ["BusinessDate"]
delivery_table_path = ""
#env_prefix = 'qa'

# ================== Create Delta Table ==================
#delivery_table_path = f"`{env_prefix}-cdh-delivery`.repoint.{table_name}"  
#AWS Credentials not available thus env_prefix returns empty temporarily we manually set it to `qa`

delivery_table_path = f"`{env_prefix}-cdh-delivery`.repoint.{table_name}"

delta_table_creator.create_unity_delta_table(
    spark, delivery_table_path, partition_columns, schema
)
# if not DeltaTable.isDeltaTable(spark, delivery_table_path):
#     delta_table_creator.create_unity_delta_table(
#         spark, delivery_table_path, partition_columns, schema
#     )
  
# ================== Delete From Delta Table ==================
transform.delete_from_unity_table(
    spark,
    delivery_table_path,
    run_date)


In [0]:
%python
# Create a DataFrame from your reference calendar table
calendar_df = spark.sql(f"SELECT * FROM `{env_prefix}-cdh`.reference.ref_calendar")

# Create a temporary view for the DataFrame
calendar_df.createOrReplaceTempView("calendar_df")

som_date = spark.sql(
f'''
SELECT 
     MAX(cal.CalendarDate) CalendarDate
FROM calendar_df AS cal
WHERE cal.IsBusinessDay = 1
AND cal.CalendarDate <= LAST_DAY(DATE_ADD(MONTH, -1, DATE '{run_date}'))
''').collect()[0][0]

referenceDateForHistorical = spark.sql(
f'''
SELECT CAST(
      CASE WHEN (SELECT DayName FROM calendar_df AS cal 
                 WHERE CalendarDate =  MAKE_DATE(year(CAST({run_date!r} AS DATE)),month(CAST({run_date!r} AS DATE)),14) ) = 'Saturday' 
            THEN (SELECT CAST(NextBusinessDay AS DATE) FROM calendar_df AS cal
                  WHERE CalendarDate =  MAKE_DATE(year(CAST({run_date!r} AS DATE)),month(CAST({run_date!r} AS DATE)),16))
            ELSE (SELECT CAST(NextBusinessDay AS DATE) FROM calendar_df AS cal 
                 WHERE CalendarDate =  MAKE_DATE(year(CAST({run_date!r} AS DATE)),month(CAST({run_date!r} AS DATE)),15)) 
      END AS DATE) AS referenceDate
''').collect()[0][0]

AssignmentDate = spark.sql(
f'''	
SELECT CalendarDate 
FROM calendar_df
WHERE CalendarDate BETWEEN DATE '{som_date}' AND DATE '{referenceDateForHistorical}' AND BusinessDayOfMonth=7
''').collect()[0][0]

SegmentationReferenceDate = spark.sql(
f'''
SELECT 
     MAX(cal.CalendarDate) CalendarDate
FROM calendar_df AS cal
WHERE cal.CalendarDate <= LAST_DAY(DATE_ADD(MONTH, -1, DATE '{run_date}'))
''').collect()[0][0]

FirstWorkingDate = spark.sql(
f'''
SELECT CAST(NextBusinessDay AS DATE) AS NextBusinessDay
FROM calendar_df
WHERE CalendarDate= '{som_date}' 
''').collect()[0][0]

In [0]:
%python
SegmentationDates = spark.sql(
f'''
SELECT B.PortfolioName
      ,B.SegmentationDate
      ,A.ExposureKeyWithPrefix  --IB 20260520
      ,MAX(A._update_timestamp) AS _update_timestamp
FROM  `{env_prefix}-ptools`.`assignments`.`offer_collection`  A
INNER JOIN (
              SELECT PortfolioName
              , MAX(BusinessDate) AS SegmentationDate
              FROM `{env_prefix}-ptools`.`assignments`.`offer_collection`
              WHERE BusinessDate <= CAST('{referenceDateForHistorical}' AS DATE) AND ExposureKeyWithPrefix IS NOT NULL
              AND PortfolioName is NOT null 
              GROUP BY PortfolioName       
           ) B
ON A.PortfolioName  = B.PortfolioName AND A.BusinessDate = B.SegmentationDate  
WHERE A.ExposureKeyWithPrefix IS NOT NULL
AND A.AssignmentStatus = 'approved'
AND  A.RelationType = 'Primary Owner'
GROUP BY B.PortfolioName
        ,B.SegmentationDate
        ,A.ExposureKeyWithPrefix 
'''
)

SegmentationDates.createOrReplaceTempView("SegmentationDates")

##### haps_portfolios

In [0]:
%python
haps_portfolios = spark.sql(
f'''
SELECT DISTINCT A.PortfolioName
FROM `{env_prefix}-reportinputfiles`.repoint.haps_repointportfolios  A
INNER JOIN (SELECT BusinessDate, MAX(IngestionTimeStamp) IngestionTimeStamp, PortfolioName 
            FROM `{env_prefix}-reportinputfiles`.repoint.haps_repointportfolios
            GROUP BY  BusinessDate, PortfolioName) B
ON A.BusinessDate = B.BusinessDate AND A.PortfolioName = B.PortfolioName AND A.IngestionTimeStamp = B.IngestionTimeStamp
WHERE A.Report = 'HAPS_CollectionsReport'
AND A.IsActive = 1 
''' 
)
haps_portfolios.createOrReplaceTempView("haps_portfolios")

##### Exposure Information

In [0]:
%python
CaseCode = spark.sql(
f'''
SELECT
	  CaseCodeKey
     ,Queue
     ,BusinessDate
FROM `{env_prefix}-cdh`.primary.casecode 
WHERE BusinessDate IN ({run_date!r},{AssignmentDate!r},{som_date!r})
and sourceSystem =  'qca'
'''
)
CaseCode.createOrReplaceTempView("CaseCode")

product = spark.sql(
f'''
SELECT 
    ProductDescription,
    ProductKey,
    SourceSystem,
    BusinessDate
FROM `{env_prefix}-cdh`.primary.product
WHERE BusinessDate IN ({run_date!r},{AssignmentDate!r},{som_date!r})
'''
)

product.createOrReplaceTempView("product")

QualcoExposure = spark.sql(
f'''
SELECT
	  CASE WHEN LEFT(e.ExposureKey,3)= por.Prefix THEN e.ExposureKey
            ELSE concat(por.Prefix,e.ExposureKey) END AS ExposureKey
    , e.BusinessDate
    , e.PortfolioName
    , CAST(e.Bucket AS INT) AS Bucket
FROM `{env_prefix}-cdh`.primary.exposure e
INNER JOIN `{env_prefix}-cdh`.primary.portfolio por
  ON por.unit_code_profits=e.portfolioKey AND e.BusinessDate=por.BusinessDate
WHERE e.BusinessDate IN (DATE({run_date!r}), DATE({AssignmentDate!r}), DATE({som_date!r}), to_date('{FirstWorkingDate}'))
AND e.PartyRelationshipType = 'Primary Owner' 
AND e.SourceSystem = 'qca'
'''
)
QualcoExposure.createOrReplaceTempView("QualcoExposure")

dexposure = spark.sql(
f'''
SELECT
	    de.ExposureKey
    , de.BusinessDate
    , de.PortfolioName
    , de.ExposureStatus
    , de.AgreementKey
    , de.CDI
    , de.AssetSegment
   -- , IFNULL(e.Bucket, de.Bucket) AS Bucket
    , IFNULL(IFNULL(e.Bucket, q.Bucket), de.Bucket) AS Bucket
    , de.OutstandingTotalBalance
    , de.GBV
    , de.OffBalance
    , de.OnBalance
    , de.DueAmount
    , de.ObjectiveStatusCode   AS ObjectiveStatus
    , de.IsActiveFlag
    , de.SubProductDescription
    , cast(de.ProductCode AS int) AS SubProductCode
    , de.HasActiveSettlementFlag
    , de.SettlementBucket
    , de.ObjectiveStatusdate
   -- , de.MasterExposureKey
    , de.ExposureKeyQCAlpha
    , de.CaseCode
    , de.Currency
    , e.ProductKey
FROM `{env_prefix}-cdh`.derivative.dexposure de
INNER JOIN haps_portfolios AS hp
ON de.PortfolioName = hp.PortfolioName
INNER JOIN `{env_prefix}-cdh`.primary.exposure e
ON e.ExposureKey = de.ExposureKey 
AND e.BusinessDate = de.BusinessDate
LEFT JOIN QualcoExposure q
ON e.ExposureKey = q.ExposureKey 
AND e.BusinessDate = q.BusinessDate
WHERE de.BusinessDate IN (DATE({run_date!r}), DATE({AssignmentDate!r}), DATE({som_date!r}), to_date('{FirstWorkingDate}'))
AND e.PartyRelationshipType = 'Primary Owner' 
AND e.SourceSystem = 'profits'
'''
)
dexposure.createOrReplaceTempView("dexposure")

master_account_step_0 = spark.sql(
f'''
SELECT DISTINCT 
         e.BusinessDate
        ,d.ExposureKey
        ,e.EntityKey              AS MasterExposureKey
        ,'Master - Slave Rltnp'   AS Relationship
        , 1                       AS RelationshipInt
        ,d.GBV
        ,d.OnBalance
        ,d.OffBalance
FROM  dexposure d
INNER JOIN `{env_prefix}-cdh`.primary.entityrelationships e
ON d.ExposureKey = e.RelatedEntityKey 
  AND d.BusinessDate = e.BusinessDate 
WHERE d.BusinessDate = {som_date!r}
UNION ALL
SELECT DISTINCT 
         e.BusinessDate
        ,d.ExposureKey
        ,e.ACCOUNT_NUMBER          AS MasterExposureKey
        ,'L1-L2 Subsidy Account'   AS Relationship
        , 3                        AS RelationshipInt
        ,d.GBV
        ,d.OnBalance
        ,d.OffBalance
FROM  dexposure d
INNER JOIN `p-cdh`.landed.profits_lns_ext_subsidy_hd e
ON d.ExposureKey = e.DEBIT_OPEN_ACCOUNT  
  AND d.BusinessDate = e.BusinessDate 
WHERE d.BusinessDate = {som_date!r}
UNION ALL 
SELECT DISTINCT
         e.BusinessDate
        ,d.ExposureKey
        ,e.RestructuringExposureKey      AS MasterExposureKey
        ,'Restructured Loan- Writeoff'   AS Relationship
        , 2                              AS RelationshipInt
        ,d.GBV
        ,d.OnBalance
        ,d.OffBalance
FROM  dexposure d
INNER JOIN `{env_prefix}-cdh`.primary.restructuringwriteoff e
ON d.ExposureKey = e.WriteOffExposureKey  
  AND d.BusinessDate = e.BusinessDate 
WHERE d.BusinessDate = {som_date!r}

'''
)
master_account_step_0.createOrReplaceTempView("master_account_step_0")

master_account_step_1 = spark.sql(
f'''
SELECT DISTINCT *
               , RANK() OVER (PARTITION BY ExposureKey ORDER BY RelationshipInt ASC, GBV DESC,MasterExposureKey DESC) AS Ranking
FROM master_account_step_0

'''
)
master_account_step_1.createOrReplaceTempView("master_account_step_1")

master_account = spark.sql(
f'''
SELECT DISTINCT *
FROM master_account_step_1
WHERE Ranking =1 
'''
)
master_account.createOrReplaceTempView("master_account")

# product_join = spark.sql(
# f'''
# SELECT DISTINCT 
# 	     d.ExposureKeyQCAlpha
#         ,e.BusinessDate
#         ,d.ExposureKey
#         ,p.ProductDescription ProductDescription
# FROM  dexposure d
# INNER JOIN `{env_prefix}-cdh`.primary.exposure e
# ON d.ExposureKey = e.ExposureKey 
#    AND d.BusinessDate = e.BusinessDate 
#    AND e.PartyRelationshipType = 'Primary Owner' 
#    AND e.SourceSystem = 'profits' 
# LEFT JOIN product p
# ON  p.ProductKey = concat(e.ProductKey)
#     AND p.SourceSystem = e.SourceSystem
# WHERE d.BusinessDate IN ({run_date!r},{AssignmentDate!r},{som_date!r})
# AND   d.PortfolioName = {run_portfolio!r}
# '''
# )
# product_join.createOrReplaceTempView("product_join")


fldt_allapplications = spark.sql(
f'''
SELECT * FROM (
select distinct unifiedLoan_AccountNumberWithPrefix,ApplicationImplementationStatus
, rank()over(partition by unifiedLoan_AccountNumberWithPrefix order by app_CurrentStepAssignmentDate desc, 
                                                                       CASE WHEN ApplicationImplementationStatus = 'Active' THEN 2 
                                                                            WHEN ApplicationImplementationStatus = 'Implemented' THEN 3 ELSE 0 END DESC) rank
from `{env_prefix}-cdh-delivery`.repoint.haps_fldtallapplications AS app
WHERE APP.ActivatedApplicationFlag = 1
 AND  app.BusinessDate = {run_date!r}
)A WHERE rank=1
'''
)
fldt_allapplications.createOrReplaceTempView("fldt_allapplications")

DPO = spark.sql(
f'''
SELECT DISTINCT unifiedLoan_AccountNumberWithPrefix--,product_ProductCategoryType, QualcoAppDisbursedDate
FROM `{env_prefix}-cdh-delivery`.repoint.haps_fldtallapplications
WHERE AccountImplementationStatus ='Implemented'
AND ActivatedApplicationFlag = 1
AND BusinessDate = {run_date!r}
AND product_ProductCategoryType LIKE '%DPO'
'''
)
DPO.createOrReplaceTempView("DPO")

In [0]:
%python
paymentschedule = spark.sql(
f'''
SELECT ExposureKey
      ,SUM(ABS(InstallmentAmount)) NextInstallmentAmount  --IB 20260520
      ,MAX(InstallmentDate)   NextInstallmentDate
FROM `{env_prefix}-cdh`.primary.paymentschedule
WHERE InstallmentType ='Installment' 
  AND InstallmentStatus = 'Active'
  AND InstallmentDate > LAST_DAY({som_date!r}) AND InstallmentDate <= LAST_DAY({run_date!r}) 
  AND businessdate={som_date!r} 
GROUP BY ExposureKey
'''
)
paymentschedule.createOrReplaceTempView("paymentschedule")

paymentscheduleExpenses = spark.sql(
f'''
SELECT ExposureKey
      ,SUM(ABS(InstallmentAmount)) ExpenseRequestAmount --IB 20260520
FROM `{env_prefix}-cdh`.primary.paymentschedule
WHERE InstallmentType ='ExpenseRequest' 
  AND InstallmentStatus = 'Active'
  AND InstallmentDate > LAST_DAY({som_date!r}) AND InstallmentDate <= LAST_DAY({run_date!r}) 
  AND businessdate={run_date!r}
GROUP BY ExposureKey
'''
)
paymentscheduleExpenses.createOrReplaceTempView("paymentscheduleExpenses")

ExchangeRates = spark.sql(
f'''
SELECT 
        E.BusinessDate  
      , E.OriginalCurrency                   
      , E.ExchangeRate                                       
FROM `{env_prefix}-cdh`.reference.exchangerates as E
where E.businessdate IN ({run_date!r},{AssignmentDate!r},{som_date!r})
'''
)
ExchangeRates.createOrReplaceTempView("ExchangeRates")

In [0]:
%python
MODS = spark.sql(
f'''
SELECT PortfolioName
      ,NewAgreementKey
      ,OldExposureKey
      ,NewExposureKey
      ,RestructuringDate
      ,BusinessDate 
FROM `{env_prefix}-cdh`.primary.restructuring
WHERE BusinessDate={run_date!r}
  AND OldExposureKey IS NOT NULL
  AND RestructuringDate IS NOT NULL 
  AND LAST_DAY(RestructuringDate)=LAST_DAY({run_date!r})
'''
)
MODS.createOrReplaceTempView("MODS")

DistinctMODS = spark.sql(
f'''
SELECT DISTINCT
       NewAgreementKey
      ,NewExposureKey
FROM MODS
'''
)
DistinctMODS.createOrReplaceTempView("DistinctMODS")

TRANSACTIONS = spark.sql(
f'''
SELECT ExposureKey
    ,SUM(col.Amount) AS TotalPayments
	,SUM(CASE WHEN col.PaymentTypeGroup IN ('Other','Cash') or col.PaymentTypeGroup is null THEN col.Amount ELSE 0 END) CashPayments
	,SUM(CASE WHEN col.PaymentTypeGroup IN ('RE Collateral Liquidation - Repossession','RE Collateral Liquidation - 3rd Party') THEN col.Amount  ELSE 0 END) 
       Liquidations 
FROM `{env_prefix}-cdh`.derivative.dcollection col
WHERE col.BusinessDate=(SELECT MAX(BusinessDate) BusinessDate FROM `{env_prefix}-cdh`.derivative.dcollection)
  AND col.FinalReason='Matched'
  AND col.ActualDate > LAST_DAY(DATEADD(MONTH, -1, {run_date!r}))		
  AND col.ActualDate <= {run_date!r}
GROUP BY ExposureKey
'''
)
TRANSACTIONS.createOrReplaceTempView("TRANSACTIONS")

Payments = spark.sql(
f'''
SELECT CASE WHEN LEFT(p.ExposureKey,3)= por.Prefix THEN p.ExposureKey
            ELSE concat(por.Prefix,p.ExposureKey) END AS ExposureKey
	,SUM(p.PaymentAmount/ COALESCE(e.ExchangeRate,1)) TotalPayments
	,SUM(CASE WHEN p.PaymentType IN ('Είσπραξη από πελάτη, κατόπιν ρύθμισης που έγινε στη Cepal.','Είσπραξη από πελάτη', 'INSURANCE CLAIM')																										THEN p.PaymentAmount/ COALESCE(e.ExchangeRate,1) ELSE 0 END) AS CashPayments 
	,SUM(CASE WHEN p.PaymentType NOT IN ('Είσπραξη από πελάτη, κατόπιν ρύθμισης που έγινε στη Cepal.','Είσπραξη από πελάτη', 'INSURANCE CLAIM')									
	      THEN p.PaymentAmount/ COALESCE(e.ExchangeRate,1) ELSE 0 END) AS Liquidations       
FROM `{env_prefix}-cdh`.primary.payment p
INNER JOIN `{env_prefix}-cdh`.primary.portfolio por
  ON por.unit_code_profits=p.portfolioKey and p.BusinessDate=por.BusinessDate
LEFT JOIN ExchangeRates e
ON p.Currency = e.OriginalCurrency
AND p.BusinessDate = e.BusinessDate
WHERE p.BusinessDate= (SELECT MAX(BusinessDate) BusinessDate FROM `{env_prefix}-cdh`.primary.payment)
  AND p.PaymentActualDate > LAST_DAY(DATEADD(MONTH, -1, {run_date!r}))		
  AND p.PaymentActualDate <= {run_date!r} 
  AND p.SourceSystem='profits'
 GROUP BY CASE WHEN LEFT(p.ExposureKey,3)= por.Prefix THEN p.ExposureKey
            ELSE concat(por.Prefix,p.ExposureKey) END
'''
)
Payments.createOrReplaceTempView("Payments")



In [0]:
%python
AccountMtD = spark.sql(
f'''
SELECT d.ExposureKey
      ,d.CDI
      ,d.Bucket
      ,d.IsActiveFlag
      ,d.ExposureStatus AS ArrearsStatusENG
      ,d.SubProductDescription
      ,c.Queue
FROM dexposure d
LEFT JOIN CaseCode c 
ON d.CaseCode = c.CaseCodeKey
AND d.BusinessDate = c.BusinessDate
WHERE d.BusinessDate = {run_date!r}
AND  d.IsActiveFlag  ='true'
'''
)
AccountMtD.createOrReplaceTempView("AccountMtD")
         
AccountAssignment = spark.sql(
f'''
SELECT d.ExposureKey
      ,d.SubProductDescription
      ,c.Queue
FROM dexposure d
LEFT JOIN CaseCode c 
ON d.CaseCode = c.CaseCodeKey
AND d.BusinessDate = c.BusinessDate
WHERE d.BusinessDate = {AssignmentDate!r}
'''
)
AccountAssignment.createOrReplaceTempView("AccountAssignment")

AccountSoMPerimeter_step1 = spark.sql(
f'''
SELECT d.ExposureKey
      ,d.BusinessDate
      ,d.PortfolioName
      ,d.ExposureStatus AS ArrearsStatusENG
      ,d.AgreementKey
      ,d.CDI
      ,d.AssetSegment
      ,d.Bucket
      ,d.OutstandingTotalBalance AS TotalLegalClaim
      ,d.GBV
      ,d.OffBalance              AS OffBalanceAmount
      ,d.OnBalance               AS OnBalanceAmount
      ,d.DueAmount               AS Delinquent
      ,d.ObjectiveStatus
      ,d.IsActiveFlag
      ,d.SubProductDescription
      ,d.SubProductCode
      ,p.ProductDescription
      ,d.HasActiveSettlementFlag
      ,d.SettlementBucket
      ,d.ObjectiveStatusdate    AS ObjectiveDate
      ,m.MasterExposureKey
      ,c.Queue
      ,d.Currency
FROM dexposure d
LEFT JOIN CaseCode c 
ON d.CaseCode = c.CaseCodeKey
AND d.BusinessDate = c.BusinessDate
LEFT JOIN product p
ON  p.ProductKey = concat(d.ProductKey)
AND p.SourceSystem = 'profits'
and p.BusinessDate = d.BusinessDate
-- LEFT JOIN product p
-- ON p.ExposureKey = d.ExposureKey
-- AND d.BusinessDate = p.BusinessDate
LEFT JOIN master_account m
ON m.ExposureKey = d.ExposureKey
AND d.BusinessDate = m.BusinessDate
WHERE d.BusinessDate = {som_date!r}
'''
)
AccountSoMPerimeter_step1.createOrReplaceTempView("AccountSoMPerimeter_step1")

AccountSoMPerimeter = spark.sql(
f'''
SELECT d.ExposureKey
      ,d.BusinessDate
      ,d.PortfolioName
      ,d.ArrearsStatusENG
      ,d.AgreementKey
      ,d.CDI
      ,d.AssetSegment
      ,d.Bucket
      ,d.TotalLegalClaim
      ,d.GBV
      ,d.OffBalanceAmount
      ,d.OnBalanceAmount
      ,d.Delinquent
      ,d.ObjectiveStatus
      ,d.IsActiveFlag
      ,d.SubProductDescription
      ,d.SubProductCode
      ,d.ProductDescription
      ,d.HasActiveSettlementFlag
      ,d.SettlementBucket
      ,d.ObjectiveDate
      ,d.MasterExposureKey
      ,d.Queue
      ,d.Currency
      ,e.ExchangeRate
      ,p.NextInstallmentAmount
      ,CAST(p.NextInstallmentDate AS DATE) NextInstallmentDate
FROM AccountSoMPerimeter_step1 d
LEFT JOIN ExchangeRates e
ON d.Currency = e.OriginalCurrency
AND e.BusinessDate = d.BusinessDate
LEFT JOIN paymentschedule p
ON p.ExposureKey = d.ExposureKey 
'''
)
AccountSoMPerimeter.createOrReplaceTempView("AccountSoMPerimeter")

#### PayersFlagAlgo

In [ ]:
%python
PayersFlagAlgo = spark.sql(  
f'''
SELECT DISTINCT   -- EK new
    PF.CDI
  , pf.Account_Number_with_Prefix
  , PF.Payer_Flag_Entity 
  , PF.Payer_Flag_Account
FROM `p-ds`.aaas.payers_flag_algorithm PF
WHERE PF.BusinessDate = '{som_date}'
'''
)

PayersFlagAlgo.createOrReplaceTempView("PayersFlagAlgo")

##### PM Tool Segmentation

In [0]:
%python
AssignmentSegmentation = spark.sql(
f'''
SELECT 
	  E.BusinessDate
    , E.ExposureKeyWithPrefix
	--, E.Borrower_Payer_Segment        	AS CustomerPayerSegment
    --, CASE WHEN E.danalytics__PayerFlagEntity ='Non_Payer' THEN 'Non Payer' ELSE E.danalytics__PayerFlagEntity END  AS CustomerPayerSegment
	,CASE WHEN pf.Payer_Flag_Entity = 'Non_Payer' THEN 'Non Payer' ELSE pf.Payer_Flag_Entity END AS CustomerPayerSegment
    , E.segment_mapped_values        	AS OfficialSegment
	, E.Segment      	                AS FinalSegment
    , E.AgreementKey
    , E.DenouncementStatus
    , CASE WHEN E.N3869Segment = 'Null' THEN NULL ELSE E.N3869Segment END AS N3869Segment
	--, E.PayerNonPayer   	             AS AccountPayerSegment
    , pf.Payer_Flag_Account AS AccountPayerSegment
    --, E.HighBalanceFlag_c                AS HighBalanceFlag
    , COALESCE(hb.HighBalanceFlag, FALSE) AS HighBalanceFlag
FROM `{env_prefix}-ptools`.`assignments`.`offer_collection` E
INNER JOIN haps_portfolios AS hp
ON e.PortfolioName = hp.PortfolioName
INNER JOIN SegmentationDates as SD ON E.BusinessDate = SD.SegmentationDate AND E.PortfolioName = SD.PortfolioName AND E._update_timestamp = SD._update_timestamp
 AND SD.ExposureKeyWithPrefix=E.ExposureKeyWithPrefix --IB 20052026
LEFT JOIN PayersFlagAlgo pf   ON E.ExposureKeyWithPrefix = pf.Account_Number_with_Prefix  -- EK new
LEFT JOIN (
    SELECT CDI, BusinessDate,
           MAX(CAST(HighBalanceFlag_c AS BOOLEAN)) AS HighBalanceFlag
    FROM `{env_prefix}-ptools`.`assignments`.`offer_collection`
    WHERE RelationType = 'Primary Owner'
      --AND AssignmentStatus = 'approved'
    GROUP BY CDI, BusinessDate
) hb ON E.CDI = hb.CDI
     AND E.BusinessDate = hb.BusinessDate
WHERE E.RelationType = 'Primary Owner'
--AND E.AssignmentStatus = 'approved'
'''
)

AssignmentSegmentation.createOrReplaceTempView("AssignmentSegmentation")


RA_Assignment = spark.sql(
f'''
SELECT *
FROM `{env_prefix}-cdh-delivery`.repoint.haps_assignmentmapping 
WHERE BusinessDate = {run_date!r}  
'''
)
RA_Assignment.createOrReplaceTempView("RA_Assignment")

In [0]:
%python
AssignmentSegmentationContract_v1 = spark.sql(
f'''
SELECT 
	  a.AgreementKey 
    , a.ExposureKeyWithPrefix
    , a.DenouncementStatus
    , a.N3869Segment
	  , a.CustomerPayerSegment
    , RAA.GModifiedAccount
    , CASE WHEN a.CustomerPayerSegment='Payer'  THEN 1 ELSE 0 END CustomerPayerSegmentInt
    , CASE WHEN a.DenouncementStatus='WRITE OFF'THEN 1 ELSE 0 END AS Woff
	  , CASE WHEN d.ObjectiveStatus LIKE '074-G%' AND RIGHT(d.ExposureKey,LEN(d.ExposureKey)-3) LIKE '400%' THEN 1
	  	     WHEN d.ObjectiveStatus LIKE '074-S%' AND d.ObjectiveStatus NOT LIKE '074-ST%' AND 
	  			 RIGHT(d.ExposureKey,LEN(d.ExposureKey)-3) LIKE '400%'									  THEN 1
	  	     ELSE 0 END AS GModified
    --,CASE WHEN RAA.GModifiedAccount = 'G_Modified' THEN 1
    --      ELSE 0
    --      END AS GModified
    , CASE WHEN d.ObjectiveStatus LIKE '074-G%' AND RIGHT(d.ExposureKey,LEN(d.ExposureKey)-3) LIKE '400%'	THEN NULL
		       WHEN d.ObjectiveStatus LIKE '074-S%' AND d.ObjectiveStatus NOT LIKE '074-ST%' AND 
			    	RIGHT(d.ExposureKey,LEN(d.ExposureKey)-3) LIKE '400%'										THEN NULL
		       ELSE a.N3869Segment END AS UpdatedN3869Segment
    , CASE WHEN d.ObjectiveStatus LIKE '074-G%' AND RIGHT(d.ExposureKey,LEN(d.ExposureKey)-3) LIKE '400%'	THEN 0
		       WHEN d.ObjectiveStatus LIKE '074-S%' AND d.ObjectiveStatus NOT LIKE '074-ST%' AND 
				RIGHT(d.ExposureKey,LEN(d.ExposureKey)-3) LIKE '400%'										THEN 0
			     WHEN a.N3869Segment = 'With Legal Restructuring'											THEN 3
			     WHEN a.N3869Segment = 'Future Court Date'														THEN 2
			     WHEN a.N3869Segment = 'Past Court Date'															THEN 1
			     ELSE NULL END AS UpdatedN3869SegmentInt
    ,  CASE WHEN ( CASE	WHEN d.ObjectiveStatus LIKE '074-G%' AND RIGHT(d.ExposureKey,LEN(d.ExposureKey)-3) LIKE '400%'	THEN 1
						            WHEN d.ObjectiveStatus LIKE '074-S%' AND d.ObjectiveStatus NOT LIKE '074-ST%' AND 
							         RIGHT(d.ExposureKey,LEN(d.ExposureKey)-3) LIKE '400%'										THEN 1
						            ELSE 0 END  ) = 1 AND a.N3869Segment IS NOT NULL							THEN 'Payer'
			      WHEN a.N3869Segment IS NOT NULL																						THEN 'L3869'
			      WHEN a.CustomerPayerSegment ='Payer'																			THEN 'Payer'
			      ELSE 'Non Payer' END AS UpdatedSegment
    ,   CASE WHEN ( CASE WHEN d.ObjectiveStatus LIKE '074-G%' AND RIGHT(d.ExposureKey,LEN(d.ExposureKey)-3) LIKE '400%'	THEN 1
						             WHEN d.ObjectiveStatus LIKE '074-S%' AND d.ObjectiveStatus NOT LIKE '074-ST%' AND 
							            RIGHT(d.ExposureKey,LEN(d.ExposureKey)-3) LIKE '400%'										THEN 1
						             ELSE 0 END  ) = 1 and a.N3869Segment IS NOT NULL											    THEN 2
			       WHEN a.N3869Segment IS NOT NULL																			   THEN 3
			       WHEN a.CustomerPayerSegment ='Payer'																		 THEN 2
			       ELSE 1 END AS  UpdatedSegmentInt
    ,    d.ObjectiveStatus  -- Check objective status is null 
    ,CASE WHEN RAA.NewOrAlphaProduct = 'New' AND RAA.GModifiedAccount='G_Modified'  THEN 1
          WHEN RAA.NewOrAlphaProduct <> 'New' AND RAA.GModifiedAccount='G_Modified'  THEN 0
          ELSE NULL END AS NewOrAlphaProduct
                											
FROM AssignmentSegmentation  a
INNER JOIN dexposure         d   ON a.ExposureKeyWithPrefix = d.ExposureKey AND CAST(d.BusinessDate AS DATE) = to_date('{FirstWorkingDate}')
LEFT  JOIN RA_Assignment     RAA ON a.ExposureKeyWithPrefix = RAA.ExposureKey
'''
)

AssignmentSegmentationContract_v1.createOrReplaceTempView("AssignmentSegmentationContract_v1")

AssignmentSegmentationContract_v2 = spark.sql(
f'''
SELECT 
	  AgreementKey 
    , MAX(Woff) AS maxwoff
    , MIN(Woff) AS minwoff
    , MAX(CustomerPayerSegmentInt) AS maxCustomerPayerSegmentInt
    , MAX(UpdatedSegmentInt)       AS maxUpdatedSegmentInt
	, MAX(UpdatedN3869SegmentInt)  AS MAXUpdatedN3869SegmentInt
    , MAX(GModified)               AS maxgmodified
    , MAX(NewOrAlphaProduct)       AS maxNewOrAlphaProduct
FROM AssignmentSegmentationContract_v1 
GROUP BY AgreementKey
'''
)

AssignmentSegmentationContract_v2.createOrReplaceTempView("AssignmentSegmentationContract_v2")

AssignmentSegmentationContract_v3 = spark.sql(
f'''
SELECT	
	   AgreementKey
	 , CASE WHEN maxwoff=1 AND minwoff=1 THEN 'Fully Woff' ELSE 'Not Fully Woff' END AS FullyWoffContract
	 , CASE WHEN maxUpdatedSegmentInt=3 AND MAXGModified=1 AND maxCustomerPayerSegmentInt=1	AND maxNewOrAlphaProduct=1 THEN '01. Payer - G Modified_New'
            WHEN maxUpdatedSegmentInt=3 AND MAXGModified=1 AND maxCustomerPayerSegmentInt=1	THEN '01. Payer - G Modified'
			WHEN maxUpdatedSegmentInt=3 AND MAXGModified=1 AND maxCustomerPayerSegmentInt=0 AND maxNewOrAlphaProduct=1	THEN '03. Non Payer - G Modified_New'
			WHEN maxUpdatedSegmentInt=3 AND MAXGModified=1 AND maxCustomerPayerSegmentInt=0	THEN '03. Non Payer - G Modified'
			WHEN maxUpdatedSegmentInt=3 AND MAXUpdatedN3869SegmentInt=3						THEN '05. L3869 - With Legal Restructuring' 
			WHEN maxUpdatedSegmentInt=3 AND MAXUpdatedN3869SegmentInt=2						THEN '06. L3869 - Future Court Date' 
			WHEN maxUpdatedSegmentInt=3 AND MAXUpdatedN3869SegmentInt=1						THEN '07. L3869 - Past Court Date' 
			WHEN maxUpdatedSegmentInt=2 AND MAXGModified=1 AND maxNewOrAlphaProduct=1 	    THEN '01. Payer - G Modified_New'
			WHEN maxUpdatedSegmentInt=2 AND MAXGModified=1								    THEN '01. Payer - G Modified'
			WHEN maxUpdatedSegmentInt=2 AND MAXGModified=0									THEN '02. Payer - Non G Modified'
			WHEN maxUpdatedSegmentInt=1 AND MAXGModified=1 AND maxNewOrAlphaProduct=1	    THEN '03. Non Payer - G Modified_New'
			WHEN maxUpdatedSegmentInt=1 AND MAXGModified=1									THEN '03. Non Payer - G Modified'
			WHEN maxUpdatedSegmentInt=1 AND MAXGModified=0									THEN '04. Non Payer - Non G Modified'
			ELSE 'Check' END AS FinalCategory
FROM AssignmentSegmentationContract_v2
'''
)

AssignmentSegmentationContract_v3.createOrReplaceTempView("AssignmentSegmentationContract_v3")

OperationalTargets = spark.sql(
f'''
SELECT     t.ReferenceDateEoM
		 , t.Portfolio
		 , cast(t.Index as int) as Index
		 , t.Category
		 , cast(t.TargetRate as decimal(19,5)) as TargetRate
		 , t.BusinessDate
		 , t.IngestionTimestamp
     , CONCAT(CASE WHEN cast(t.Index as int) < 10 THEN CONCAT('0', CAST(Index AS VARCHAR(1))) ELSE CAST(Index AS VARCHAR(2)) END, '_', Category) AS MatchingCategory
FROM `{env_prefix}-reportinputfiles`.repoint.haps_collectionsoperationaltargets t
INNER JOIN (SELECT MAX(IngestionTimeStamp) IngestionTimeStamp, BusinessDate FROM `{env_prefix}-reportinputfiles`.repoint.haps_collectionsoperationaltargets GROUP BY BusinessDate) its ON its.IngestionTimeStamp = t.IngestionTimeStamp AND its.BusinessDate = t.BusinessDate
WHERE last_day(ReferenceDateEoM) = last_day({som_date!r}) 
  AND t.BusinessDate = (
        SELECT MAX(BusinessDate) MaxDate 
        FROM `{env_prefix}-reportinputfiles`.repoint.haps_collectionsoperationaltargets 
        WHERE last_day(ReferenceDateEoM) = last_day({som_date!r})
    )
'''
)

OperationalTargets.createOrReplaceTempView("OperationalTargets")

In [0]:
%python
Law3869 = spark.sql(
f'''
SELECT CDI
     , InLawFlag L3869Flag
     , CASE WHEN LawStatus = 'Έφεση (Πρωτόδικη Απόφ. Δεκτή)' THEN 'Positive Court Decision' 
            ELSE CalcLawStatus END AS ApplicationStatus
FROM `{env_prefix}-cdh`.primary.underlaw
WHERE InLawFlag=1 AND UnderLawType='N3869'
  AND businessdate={som_date!r} 
'''
)

Law3869.createOrReplaceTempView("Law3869")

QueueMapping = spark.sql(
f'''
SELECT DISTINCT E.*
FROM `{env_prefix}-reportinputfiles`.repoint.haps_queuesmapping as E
INNER JOIN (SELECT MAX(IngestionTimeStamp) IngestionTimeStamp, BusinessDate FROM `{env_prefix}-reportinputfiles`.repoint.haps_queuesmapping GROUP BY BusinessDate) its ON E.IngestionTimeStamp = its.IngestionTimeStamp and E.BusinessDate = its.BusinessDate
WHERE E.BusinessDate IN 
		(
		SELECT MAX(BusinessDate) MaxDate FROM `{env_prefix}-reportinputfiles`.repoint.haps_queuesmapping 
		)
'''
)

QueueMapping.createOrReplaceTempView("QueueMapping")



In [0]:
%python
TEMP1_Step1 = spark.sql(
f'''
SELECT DISTINCT
			   SOM.ExposureKey
			  ,SOM.BusinessDate
			  ,SOM.PortfolioName AS Portfolio
			  ,RIGHT(SOM.ExposureKey, LEN(SOM.ExposureKey)-3 )	 AS AccountNumberNoPrefix
			  ,SOM.AgreementKey                                  AS AgreementNumber
			  ,SOM.CDI
			  ,SOM.AssetSegment
			  ,SOM.Bucket       AS SoMBucket
			  ,SOM.TotalLegalClaim	/COALESCE(SOM.ExchangeRate,1)	AS SoMTotalLegalClaimEur
			  ,SOM.GBV/COALESCE(SOM.ExchangeRate,1)				        AS SoMGBVEur
			  ,SOM.Delinquent										AS SoMDelinquent
			  ,SOM.OnBalanceAmount
			  ,SOM.OffBalanceAmount
			  ,SOM.ObjectiveStatus
			  ,CASE WHEN SOM.IsActiveFlag <> 'true'	     THEN 'OffBalance' 
				    	WHEN SOM.ArrearsStatusENG NOT LIKE '%Writen Off%'	
                     AND SOM.SubProductDescription	NOT LIKE 'PRF ΤΟΚΟΙ%'	
                     AND SOM.ProductDescription		NOT LIKE 'ΥΠΟ ΑΙΡΕΣΗ ΔΙΑΓΡΑΦΗ - RET - ΡΥΘΜΙΣΕΙΣ' 
                     AND SOM.GBV>0                  THEN 'Onbalance'
					    ELSE 'OffBalance' END AS AccountBalanceFlag
			  ,CASE WHEN SOM.IsActiveFlag <> 'true'	                                                           THEN 'Closed' 
					    WHEN SOM.ArrearsStatusENG NOT LIKE '%Writen Off%'	
                     AND  SOM.SubProductDescription NOT LIKE 'PRF ΤΟΚΟΙ%'	
                     AND  SOM.ProductDescription NOT LIKE 'ΥΠΟ ΑΙΡΕΣΗ ΔΙΑΓΡΑΦΗ - RET - ΡΥΘΜΙΣΕΙΣ' 
                     AND  SOM.GBV>0                                                                     THEN 'Onbalance'
					    WHEN SOM.IsActiveFlag <> 'true'		                                                        THEN 'Closed'
					    WHEN SOM.SubProductDescription	LIKE 'PRF ΤΟΚΟΙ%'									                        THEN 'PRF ΤΟΚΟΙ Ν3869 ΥΠΟ ΑΝΑΣΤΟΛΗ'
					    WHEN SOM.ProductDescription		LIKE 'ΥΠΟ ΑΙΡΕΣΗ ΔΙΑΓΡΑΦΗ - RET - ΡΥΘΜΙΣΕΙΣ'		            THEN 'ΥΠΟ ΑΙΡΕΣΗ ΔΙΑΓΡΑΦΗ'
					    WHEN SOM.ArrearsStatusENG		LIKE '%Writen Off%'									                          THEN 'Writeoff'
					    ELSE 'OffBalance' END AS AccountType
			  ,SOM.SubProductDescription
			  ,SOM.SubProductCode
			  ,SOM.ProductDescription
			  ,SOM.IsActiveFlag										AS SoMIsActiveFlag
			  ,SOM.NextInstallmentDate								AS NextInstallmentDate
			  ,SOM.NextInstallmentAmount						    AS NextInstallmentAmount
			  ,CASE WHEN SOM.ArrearsStatusENG IN ('Normal','Overdue')					 
                     AND SOM.IsActiveFlag = 'true'											 
                     AND date_format(SOM.NextInstallmentDate, 'yyyy-MM') = date_format({run_date!r}, 'yyyy-MM')		 
                     AND (SOM.HasActiveSettlementFlag = 'false' OR SOM.HasActiveSettlementFlag IS NULL)
                    THEN COALESCE(SOM.NextInstallmentAmount,0)
					    ELSE 0 END NextInstallmentAmountCalculated
			  ,SOM.HasActiveSettlementFlag
			  ,SOM.SettlementBucket
			  ,SOM.ObjectiveDate
			  ,CASE WHEN SOM.IsActiveFlag <> 'true'	                                              THEN 'Closed'
              WHEN RAA.AssignmentChannel = 'Outsourcing Collection'
               AND SOM.GBV = SOM.Delinquent AND SOM.Bucket>=4                               THEN 'Denounced'
					    WHEN SOM.ArrearsStatusENG IN ('Normal','Overdue') AND SOM.Bucket= 0	          THEN 'Current'
					    WHEN SOM.ArrearsStatusENG IN ('Overdue') AND SOM.Bucket<=3					  THEN CAST(SOM.Bucket AS STRING)
				      WHEN SOM.ArrearsStatusENG IN ('Normal','Overdue')							        THEN '4+'
					    WHEN SOM.ArrearsStatusENG IN ('Define Delay')								  THEN 'Denounced'
					    WHEN SOM.ArrearsStatusEng IN ('Writen Off')									  THEN 'Write-off'
					END	BucketGroup
			   --,ISNULL(SOM.Delinquent,0) + CASE WHEN SOM.ArrearsStatusENG IN ('Normal','Overdue') AND 
			   --					                  SOM.IsActiveFlag=1 AND EOMONTH(SOM.NextInstallmentDate)=EOMONTH(@MtDReferenceDate)		 AND
			   --					                  ISNULL(SOM.HasActiveSettlementFlag,0)=0 THEN ISNULL(SOM.NextInstallmentAmount,0)  
			   --							     ELSE 0 END	AS EoMDelinquentAmount
         ,COALESCE(SOM.Delinquent,0) + COALESCE(SOM.NextInstallmentAmount,0) AS EoMDelinquentAmount
			   ,MasterExposureKey MasterAccount
         ,CASE WHEN SOM.IsActiveFlag = 'true'	 THEN RAA.QueueDescriptionFinal 
               ELSE NULL END AS QueueDescription
			   ,CASE WHEN SOM.IsActiveFlag = 'true'  THEN RAA.CollectionIntramonthAssignment 
                ELSE NULL END AS IntramonthFlag		
 			-- ,CASE WHEN RIGHT(SOM.ExposureKey, LEN(SOM.ExposureKey)-3 ) LIKE '400%'				
			--		  AND (SOM.SubProductDescription LIKE 'G%' OR SOM.SubProductDescription LIKE'S%' OR SOM.SubProductDescription LIKE '0000%')	
			--		  AND SOM.ProductDescription NOT LIKE '%ΑΙΡΕΣΗ%'
			--		  AND SOM.IsActiveFlag = 'true'				THEN 'Modified'
			--		 WHEN SOM.IsActiveFlag = 'true'				THEN 'NonModified'
			--		 ELSE NULL END	AS GAccountFlag 
          ,CASE WHEN RAA.GModifiedAccount = 'G_Modified' THEN 'Modified'
                ELSE 'NonModified' -- NULL 
                END AS GAccountFlag
     	  ,RAA.AssignmentChannel
		  ,RAA.AssignmentTeam
		  ,RAA.AssignmentAgent  
 		  ,MTD.Bucket																												AS SourceBucketMtD
		  ,CASE WHEN	SOM.IsActiveFlag = 'true'	AND (MTD.IsActiveFlag <> 'true' OR MTD.IsActiveFlag IS NULL)	 THEN 'Current'
			    WHEN	(MTD.IsActiveFlag <> 'true' OR MTD.IsActiveFlag IS NULL)														     THEN 'Closed'
				WHEN	MTD.Bucket= 0																		                                         THEN 'Current'
				 WHEN	MTD.ArrearsStatusENG IN ('Overdue','Normal') AND MTD.Bucket<=3	                         THEN CAST(MTD.Bucket AS VARCHAR(20))
				 WHEN	MTD.ArrearsStatusENG IN ('Overdue','Normal')							                               THEN '4+'
				 WHEN	MTD.ArrearsStatusENG IN ('Define Delay')													                       THEN 'Denounced'
				 WHEN	MTD.ArrearsStatusENG IN ('Writen Off')													                         THEN 'Write-off'
							END	AS BucketGroupMtD	     
   		  ,SOM.ArrearsStatusENG		AS SoMArrearsStatusENG
		  ,CASE WHEN SOM.Queue LIKE '%PJ%' THEN 1 ELSE 0 END AS PJ 
          ,SOM.GBV 
          ,SOM.Queue Queues 
          ,RAA.QueueDescriptionFinal   
						 
FROM AccountSoMPerimeter  SOM
LEFT JOIN AccountMtD											    MTD ON SOM.ExposureKey=MTD.ExposureKey
LEFT JOIN RA_Assignment										    RAA ON SOM.ExposureKey=RAA.ExposureKey
'''
)

TEMP1_Step1.createOrReplaceTempView("TEMP1_Step1")

In [0]:
%python
TEMP1_Step2 = spark.sql(
f'''
SELECT DISTINCT
			   SOM.ExposureKey
			  ,SOM.BusinessDate
			  ,SOM.Portfolio
			  ,SOM.AccountNumberNoPrefix
			  ,SOM.AgreementNumber
			  ,SOM.CDI
			  ,SOM.AssetSegment
			  ,SOM.SoMBucket
			  ,SOM.SoMTotalLegalClaimEur
			  ,SOM.SoMGBVEur
			  ,SOM.SoMDelinquent
			  ,SOM.OnBalanceAmount
			  ,SOM.OffBalanceAmount
			  ,SOM.ObjectiveStatus
			  ,SOM.AccountBalanceFlag
			  ,SOM.AccountType
			  ,SOM.SubProductDescription
			  ,SOM.SubProductCode
			  ,SOM.ProductDescription
			  ,SOM.SoMIsActiveFlag
			  ,SOM.NextInstallmentDate
			  ,SOM.NextInstallmentAmount
			  ,SOM.NextInstallmentAmountCalculated
			  ,SOM.HasActiveSettlementFlag
			  ,SOM.SettlementBucket
			  ,SOM.ObjectiveDate
			  ,SOM.BucketGroup
			  ,SOM.EoMDelinquentAmount
			  ,SOM.MasterAccount
			  ,CASE WHEN SOM.SoMIsActiveFlag= 'true' THEN ASG.FinalSegment		
					    ELSE NULL END	AS FinalSegment
			  ,CASE WHEN SOM.SoMIsActiveFlag= 'true' THEN ASG.OfficialSegment	
					    ELSE NULL END	AS OfficialSegment
			  ,CASE WHEN SOM.SoMIsActiveFlag= 'true' AND ASG.CustomerPayerSegment IS NOT NULL	THEN ASG.CustomerPayerSegment
					    WHEN SOM.SoMIsActiveFlag= 'true' AND							 
						       SOM.SoMArrearsStatusENG NOT LIKE '%Writen Off%'	AND 						 
						       SOM.SubProductDescription NOT LIKE 'PRF ΤΟΚΟΙ%'	AND 						 
						       SOM.ProductDescription	NOT LIKE 'ΥΠΟ ΑΙΡΕΣΗ ΔΙΑΓΡΑΦΗ - RET - ΡΥΘΜΙΣΕΙΣ' AND 
						       SOM.GBV>0																		 THEN 'Non Payer'
					    ELSE Null END AS CustomerPayerSegment
			  ,	CASE WHEN SOM.SoMIsActiveFlag= 'true' AND ASG.AccountPayerSegment IS NOT NULL THEN ASG.AccountPayerSegment
					 WHEN SOM.SoMIsActiveFlag= 'true' AND							 
						  SOM.SoMArrearsStatusENG NOT LIKE '%Writen Off%'	AND 						 
						  SOM.SubProductDescription	NOT LIKE 'PRF ΤΟΚΟΙ%'	AND 						 
						  SOM.ProductDescription NOT LIKE 'ΥΠΟ ΑΙΡΕΣΗ ΔΙΑΓΡΑΦΗ - RET - ΡΥΘΜΙΣΕΙΣ' AND 
						  SOM.GBV>0																		 THEN 'Non Payer'
					 ELSE Null END	AS AccountPayerSegment     
              ,SOM.QueueDescription
			  ,SOM.IntramonthFlag	
			  ,CASE WHEN ACS.SubProductDescription LIKE 'PRF_ΔΙΚΑΣΤΙΚΗ%'						OR										 
						 ACS.SubProductDescription LIKE 'PRF ΔΙΚΑΣΤΙΚΗ%'						THEN 'Positive Court Decision'  					
					WHEN SOM.SubProductDescription LIKE 'PRF_ΔΙΚΑΣΤΙΚΗ%'						OR										 
						 SOM.SubProductDescription LIKE 'PRF ΔΙΚΑΣΤΙΚΗ%'						THEN 'Positive Court Decision'
					WHEN (ACS.Queue LIKE '%3869%' AND ACS.Queue NOT LIKE '%Modified%' AND ACS.Queue NOT LIKE '%modified%')		OR 
						 (ACS.Queue LIKE '%4605%' AND ACS.Queue NOT LIKE '%Modified%' AND ACS.Queue NOT LIKE '%modified%')		THEN 'Yes' 
					WHEN (SOM.Queues LIKE '%3869%' AND SOM.Queues NOT LIKE '%Modified%' AND SOM.Queues NOT LIKE '%modified%')		OR 
						 (SOM.Queues LIKE '%4605%' AND SOM.Queues NOT LIKE '%Modified%' AND SOM.Queues NOT LIKE '%modified%')		THEN 'Yes' 
					ELSE 'No' END L3869Queues	     	
 			  ,SOM.GAccountFlag 
     		,SOM.AssignmentChannel
			  ,SOM.AssignmentTeam
			  ,SOM.AssignmentAgent  
 			  ,SOM.SourceBucketMtD
			  ,SOM.BucketGroupMtD	     
   			,SOM.SoMArrearsStatusENG
			  ,SOM.PJ    
        ,SOM.QueueDescriptionFinal 
        ,ASG.HighBalanceFlag          																	 
FROM TEMP1_Step1  SOM
LEFT JOIN AssignmentSegmentation							ASG ON SOM.ExposureKey=ASG.ExposureKeyWithPrefix
LEFT JOIN AccountAssignment									ACS ON SOM.ExposureKey=ACS.ExposureKey
'''
)

TEMP1_Step2.createOrReplaceTempView("TEMP1_Step2")

In [0]:
%python
TEMP1 = spark.sql(
f'''
SELECT DISTINCT
			   SOM.ExposureKey
			  ,SOM.BusinessDate
			  ,SOM.Portfolio
			  ,SOM.AccountNumberNoPrefix
			  ,SOM.AgreementNumber
			  ,SOM.CDI
			  ,SOM.AssetSegment
			  ,SOM.SoMBucket
			  ,SOM.SoMTotalLegalClaimEur
			  ,SOM.SoMGBVEur
			  ,SOM.SoMDelinquent
			  ,SOM.OnBalanceAmount
			  ,SOM.OffBalanceAmount
			  ,SOM.ObjectiveStatus
			  ,SOM.AccountBalanceFlag
			  ,SOM.AccountType
			  ,SOM.SubProductDescription
			  ,SOM.SubProductCode
			  ,SOM.ProductDescription
			  ,SOM.SoMIsActiveFlag
			  ,SOM.NextInstallmentDate
			  ,SOM.NextInstallmentAmount
			  ,SOM.NextInstallmentAmountCalculated
			  ,SOM.HasActiveSettlementFlag
			  ,SOM.SettlementBucket
			  ,SOM.ObjectiveDate
			  ,SOM.BucketGroup
			  ,SOM.EoMDelinquentAmount
			  ,SOM.MasterAccount
			  ,SOM.FinalSegment
			  ,SOM.OfficialSegment
			  ,SOM.CustomerPayerSegment
			  ,CASE WHEN SOM.AccountPayerSegment='Non_Payer' THEN 'Non Payer' ELSE SOM.AccountPayerSegment END AS AccountPayerSegment    
              ,SOM.QueueDescription
			  ,SOM.IntramonthFlag	
			  ,SOM.L3869Queues	     	
 			  ,SOM.GAccountFlag 
			  ,LAW.L3869Flag	AS SoML3869Flag
			  ,LAW.ApplicationStatus AS SoML3869Status      
     		  ,SOM.AssignmentChannel
			  ,SOM.AssignmentTeam
			  ,SOM.AssignmentAgent  
 			  ,QUE.MasterSegment AS MasterSegmentQueue
			  ,QUE.AssignmentStatus	AS AssignmentStatusQueue
			  ,QUE.Channel	AS ChannelQueue
			  ,QUE.CourseOfAction	AS CourseOfActionQueue    
 			  ,SOM.SourceBucketMtD
			  ,SOM.BucketGroupMtD	  
			  ,CON.FinalCategory AS ContractLevelAssignmentCategory
   			  ,SOM.SoMArrearsStatusENG
			  ,SOM.PJ 
              ,SOM.HighBalanceFlag              																	 
FROM TEMP1_Step2  SOM
LEFT JOIN Law3869											          LAW ON SOM.CDI=LAW.CDI
LEFT JOIN AssignmentSegmentationContract_v3					          CON ON CON.AgreementKey=SOM.AgreementNumber
LEFT JOIN QueueMapping										          QUE ON SOM.QueueDescriptionFinal=QUE.QueueDescription 
																			-- AND CASE WHEN SOM.Portfolio LIKE 'Galaxy II%' THEN 'GalaxyRetail'
																			-- 		 WHEN SOM.Portfolio LIKE 'Orion%' THEN 'GalaxyRetail'
																			-- 		 WHEN SOM.Portfolio = 'Cosmos Retail' THEN 'CosmosRetail'
																			-- 	END = QUE.Portfolio

'''
)

TEMP1.createOrReplaceTempView("TEMP1")

In [0]:
%python
TEMP3 = spark.sql(
f'''
	SELECT *
		 ,CASE WHEN AccountType<>'Onbalance'	                               THEN 'OffBalance'
			   WHEN PJ=1													                             THEN 'Payers_PJ_Assigned'
			   WHEN IntramonthFlag=1 AND L3869Queues='No'																		
                AND AccountBalanceFlag='Onbalance'							         THEN 'Collection_Intramonth'
			   WHEN IntramonthFlag=2 AND L3869Queues='No'																		
                AND AccountBalanceFlag='Onbalance'							         THEN 'Collection_Intramonth_2'
			   WHEN IntramonthFlag=1 AND GAccountFlag='Modified'																
                AND  AccountBalanceFlag='Onbalance'							         THEN 'Collection_Intramonth'
			   WHEN IntramonthFlag=2 AND GAccountFlag='Modified'																
                AND  AccountBalanceFlag='Onbalance'											 THEN 'Collection_Intramonth_2'
			   WHEN AssignmentStatusQueue='Collection'	AND  SoMBucket=0																			
                AND IntramonthFlag=0 AND L3869Queues='No'																		
                AND AssignmentChannel IN ('Outsourcing Collection',	'FAs')					THEN 'Collection'
			   WHEN BucketGroup ='Current' AND  L3869Queues='No'											   THEN 'Current'				 
			   WHEN AssignmentStatusQueue='Collection' AND 
					L3869Queues='No' AND AssignmentChannel IN ('Outsourcing Collection',	'FAs')						   THEN 'Collection'
			   WHEN CustomerPayerSegment='Payer' AND AssignmentStatusQueue NOT LIKE 'rest%'												
                AND L3869Queues='No'																		   THEN 'Payers_Unhandled'
			   WHEN CustomerPayerSegment='Non Payer' AND COALESCE(AssignmentStatusQueue,'') NOT LIKE 'rest%'										
                AND L3869Queues='No'																		   THEN 'Non_Payers_Not_Assigned'
			   WHEN AssignmentStatusQueue ILIKE 'rest%' AND L3869Queues='No'									   THEN 'Non_Payers_Assigned'
			   WHEN IntramonthFlag=1 AND GAccountFlag='NonModified'															
                AND  QueueDescription ILIKE 'Oristikes_Apofaseis%' 
                AND AccountBalanceFlag='Onbalance'															   THEN '3869_Positive_Court_Collection_Intramonth'
			   WHEN IntramonthFlag=2 AND GAccountFlag='NonModified'															
                AND QueueDescription ILIKE 'Oristikes_Apofaseis%' 
                AND AccountBalanceFlag='Onbalance'													           THEN '3869_Positive_Court_Collection_Intramonth_2'
			   WHEN AssignmentChannel IN ('Outsourcing Collection',	'FAs')											
                AND AssignmentStatusQueue ilike 'collection'	AND SoMBucket=0																			
                AND IntramonthFlag=0 AND L3869Queues='Positive Court Decision'								   THEN '3869_Positive_Court_Collection'
			   WHEN BucketGroup ='Current' AND  AssignmentStatusQueue not like 'collection'											
                AND L3869Queues='Positive Court Decision'													   THEN '3869_Positive_Court_Current'
			   WHEN AssignmentChannel IN ('Outsourcing Collection',	'FAs') AND AssignmentStatusQueue ilike 'collection'												
                AND L3869Queues='Positive Court Decision'													   THEN '3869_Positive_Court_Collection'
			   WHEN IntramonthFlag=1 AND L3869Queues='Positive Court Decision'													
                AND AccountBalanceFlag='Onbalance'															   THEN '3869_Positive_Court_Collection_Intramonth'
			   WHEN IntramonthFlag=2 AND L3869Queues='Positive Court Decision'													
                AND AccountBalanceFlag='Onbalance'															   THEN '3869_Positive_Court_Collection_Intramonth_2'
			   WHEN L3869Queues='Positive Court Decision'															THEN '3869_Positive_Court_Unhandled'		
			   WHEN AssignmentChannel IN ('Outsourcing Collection',	'FAs') AND AssignmentStatusQueue ilike 'collection'												
                AND L3869Queues='Yes'																				THEN 'Collection'	
			   WHEN L3869Queues='Yes'																				       THEN '3869_Rest'					
			   END AS category2
    FROM TEMP1
  '''
)

TEMP3.createOrReplaceTempView("TEMP3")


##### Rest Calculations

In [0]:
%python
TEMP4 = spark.sql(
f'''	
    SELECT
    	  	A.*
     		,	IFNULL(T.TotalPayments,0) + IFNULL(CAST( A.SoMGBVEur AS FLOAT) / CASE WHEN SUM( A.SoMGBVEur) OVER (PARTITION BY A.AgreementNumber) > 0 THEN SUM( A.SoMGBVEur) OVER (PARTITION BY A.AgreementNumber) END, 0) * IFNULL(T_n.TotalPayments,0)	AS TotalTransactions
             ,	IFNULL(T.CashPayments,0) + IFNULL(CAST( A.SoMGBVEur AS FLOAT) / CASE WHEN SUM( A.SoMGBVEur) OVER (PARTITION BY A.AgreementNumber) > 0 THEN SUM( A.SoMGBVEur) OVER (PARTITION BY A.AgreementNumber) END, 0) * IFNULL(T_n.CashPayments,0)			AS CashTransactions
			 ,	IFNULL(T.Liquidations,0) + IFNULL(CAST( A.SoMGBVEur AS FLOAT) / CASE WHEN SUM( A.SoMGBVEur) OVER (PARTITION BY A.AgreementNumber) > 0 THEN SUM( A.SoMGBVEur) OVER (PARTITION BY A.AgreementNumber) END, 0) * IFNULL(T_n.Liquidations,0)		AS LiquidationTransactions
           
            ,	IFNULL(P.TotalPayments,0) + IFNULL(CAST( A.SoMGBVEur AS FLOAT) / CASE WHEN SUM( A.SoMGBVEur) OVER (PARTITION BY A.AgreementNumber) > 0 THEN SUM( A.SoMGBVEur) OVER (PARTITION BY A.AgreementNumber) END, 0) * IFNULL(P_n.TotalPayments,0)	AS TotalPayments
			 ,	IFNULL(P.CashPayments,0) + IFNULL(CAST( A.SoMGBVEur AS FLOAT) / CASE WHEN SUM( A.SoMGBVEur) OVER (PARTITION BY A.AgreementNumber) > 0 THEN SUM( A.SoMGBVEur) OVER (PARTITION BY A.AgreementNumber) END, 0) * IFNULL(P_n.CashPayments,0)		AS CashPayments
			 ,	IFNULL(CAST( A.SoMGBVEur AS FLOAT) / CASE WHEN SUM( A.SoMGBVEur) OVER (PARTITION BY A.AgreementNumber) > 0 THEN SUM( A.SoMGBVEur) OVER (PARTITION BY A.AgreementNumber) END, 0) * IFNULL(P_n.CashPayments,0)		AS NewCashPayments
			 ,	IFNULL(P.Liquidations,0) + IFNULL(CAST( A.SoMGBVEur AS FLOAT) / CASE WHEN SUM( A.SoMGBVEur) OVER (PARTITION BY A.AgreementNumber) > 0 THEN SUM( A.SoMGBVEur) OVER (PARTITION BY A.AgreementNumber) END, 0) * IFNULL(P_n.Liquidations,0)		AS Liquidations
			 , MIN(CASE WHEN A.BucketGroup='Write-off' THEN 1 ELSE 0 END) OVER (PARTITION BY A.AgreementNumber) AS FullyWoff
			 , MIN(CASE WHEN A.BucketGroup='Closed' THEN 1 ELSE 0 END) OVER (PARTITION BY A.AgreementNumber) AS FullyClosed	
    FROM TEMP3 A
	LEFT  JOIN TRANSACTIONS	T	ON T.ExposureKey = A.ExposureKey
	LEFT  JOIN (SELECT M.NewAgreementKey, SUM(T.TotalPayments) AS TotalPayments, SUM(T.CashPayments) AS CashPayments, SUM(T.Liquidations) AS Liquidations
 				FROM TRANSACTIONS T
				INNER JOIN DistinctMODS M ON M.NewExposureKey=T.ExposureKey
    			GROUP BY M.NewAgreementKey
       			) 			T_n	ON T_n.NewAgreementKey = A.AgreementNumber
	LEFT  JOIN Payments		P	ON P.ExposureKey = A.ExposureKey
	LEFT  JOIN (SELECT M.NewAgreementKey, SUM(P.TotalPayments) AS TotalPayments, SUM(P.CashPayments) AS CashPayments, SUM(P.Liquidations) AS Liquidations
 				FROM Payments P
				INNER JOIN DistinctMODS M ON M.NewExposureKey=P.ExposureKey
    			GROUP BY M.NewAgreementKey
       			) 			P_n	ON P_n.NewAgreementKey = A.AgreementNumber
'''
) 
TEMP4.createOrReplaceTempView("TEMP4") 

In [0]:
%python
TEMP5_Step0 = spark.sql(
f'''	
    SELECT DISTINCT
			 BusinessDate AS SoMReferenceDate
			,{run_date!r}  AS ReferenceDate
			,Portfolio
			,AccountBalanceFlag
			,AccountType
			,ExposureKey
			,AccountNumberNoPrefix
			,GAccountFlag
			,AgreementNumber
			,SoMIsActiveFlag
			,CDI
		  ,AssetSegment
		  ,SubProductDescription
		  ,ProductDescription
		  ,CAST(SoMBucket AS STRING) AS SoMBucket
		  ,BucketGroup AS SoMBucketGroup
		  ,' ' AS SoMArrearsStatus
		  ,HasActiveSettlementFlag
		  ,NextInstallmentDate
		  ,NextInstallmentAmount
		  ,SoMTotalLegalClaimEur
		  ,SoMGBVEur
		  ,SoMDelinquent
		  ,EoMDelinquentAmount
		  ,ObjectiveStatus
			,ObjectiveDate
			,QueueDescription
			,AssignmentChannel
			,AssignmentTeam
			,AssignmentAgent
			,IntramonthFlag
			,FinalSegment
			,OfficialSegment
			,L3869Queues as L3869Flag
			,CustomerPayerSegment
			,CashTransactions
			,LiquidationTransactions
			,CashPayments
			,Liquidations
			,NewCashPayments
			,category2 as ReportingCategory
			,CAST (SourceBucketMtD AS STRING) AS MtDMBucket
			,BucketGroupMtD AS MtDBucketGroup
			,MasterAccount
			,FullyWoff
			,FullyClosed
			,SoML3869Flag
			,SoML3869Status
			,ContractLevelAssignmentCategory
			,AccountPayerSegment
			,OnBalanceAmount
			,OffBalanceAmount
			,SoMArrearsStatusENG
      		,HighBalanceFlag  
	FROM TEMP4 A
	-- LEFT JOIN FullyWoff	    B ON A.AgreementNumber=B.AgreementNumber
	-- LEFT JOIN FullyClosed	C ON A.AgreementNumber=C.AgreementNumber
           '''
) 
TEMP5_Step0.createOrReplaceTempView("TEMP5_Step0") 

##### Slave Allocation

In [0]:
%python
PaymentsTransactionsSlaveAllocation_Step0 = spark.sql(
f'''	
    SELECT A.ExposureKey
                    ,A.MasterAccount
                    ,A.CashPayments AS CashPaymentsSlave
                    ,B.CashPayments AS CashPaymentsMaster
                    ,A.CashTransactions AS CashTransactionsSlave
                    ,B.CashTransactions AS CashTransactionsMaster -- Ammendmend FM 30102025
	FROM TEMP4 A
    LEFT JOIN TEMP4 B ON A.MasterAccount=B.ExposureKey
	 WHERE A.MasterAccount IS NOT NULL
	   AND A.FullyWoff=0
	   AND A.FullyClosed=0
	   AND A.AccountBalanceFlag='OffBalance' 
	   AND A.CashPayments>0
            '''
) 
PaymentsTransactionsSlaveAllocation_Step0.createOrReplaceTempView("PaymentsTransactionsSlaveAllocation_Step0")     

PaymentsTransactionsSlaveAllocation = spark.sql(
f'''	
    SELECT MasterAccount
          ,CashPaymentsMaster -- FM 29012026 instead of COALESCE(SUM(CashPaymentsMaster),0) CashPaymentsMaster
          ,COALESCE(SUM(CashPaymentsSlave),0) CashPaymentsSlave
          ,CashTransactionsMaster -- FM 29012026 instead of COALESCE(SUM(CashTransactionsMaster),0) CashTransactionsMaster
          ,COALESCE(SUM(CashTransactionsSlave),0) CashTransactionsSlave
	FROM PaymentsTransactionsSlaveAllocation_Step0
	GROUP BY MasterAccount
        , CashPaymentsMaster, CashTransactionsMaster -- FM 29012026
            '''
) 
PaymentsTransactionsSlaveAllocation.createOrReplaceTempView("PaymentsTransactionsSlaveAllocation") 



In [0]:
%python
TEMP5 = spark.sql(
f'''	
    SELECT distinct 
			 SoMReferenceDate
			,ReferenceDate
			,Portfolio
			,AccountBalanceFlag
			,AccountType
			,ExposureKey
		,AccountNumberNoPrefix
			,GAccountFlag
			,AgreementNumber
			,SoMIsActiveFlag
			,CDI
		    ,AssetSegment
		    ,SubProductDescription
		    ,ProductDescription
		    ,SoMBucket
		    ,SoMBucketGroup
		    ,SoMArrearsStatus
		    ,HasActiveSettlementFlag
		    ,NextInstallmentDate
		    ,NextInstallmentAmount
		    ,SoMTotalLegalClaimEur
		    ,SoMGBVEur
		    ,SoMDelinquent
		    ,EoMDelinquentAmount
		    ,ObjectiveStatus
		    ,ObjectiveDate
			,QueueDescription
			,AssignmentChannel
			,AssignmentTeam
			,AssignmentAgent
			,IntramonthFlag
			,FinalSegment
			,OfficialSegment
			,L3869Flag
			,CustomerPayerSegment
            ,CASE WHEN B.MasterAccount IS NOT NULL THEN B.CashTransactionsMaster+B.CashTransactionsSlave 
                  WHEN D.MasterAccount IS NOT NULL THEN 0
                  ELSE A.CashTransactions END AS CashTransactions -- Ammendmend FM 30102025
			,LiquidationTransactions
			,CASE WHEN B.MasterAccount IS NOT NULL THEN B.CashPaymentsMaster+B.CashPaymentsSlave 
                  WHEN D.MasterAccount IS NOT NULL THEN 0
                  ELSE A.CashPayments END AS CashPayments
			,Liquidations
			,NewCashPayments
			,ReportingCategory
			,MtDMBucket
			,MtDBucketGroup
			,A.MasterAccount                         
			,FullyWoff
			,FullyClosed
			,SoML3869Flag
			,SoML3869Status
			,ContractLevelAssignmentCategory
			,AccountPayerSegment
			,OnBalanceAmount
			,OffBalanceAmount
			,SoMArrearsStatusENG
      		,HighBalanceFlag  
	FROM TEMP5_Step0 A
	LEFT JOIN PaymentsTransactionsSlaveAllocation	    B ON A.ExposureKey=B.MasterAccount
    LEFT JOIN PaymentsTransactionsSlaveAllocation       D ON A.MasterAccount=D.MasterAccount
	--LEFT JOIN TransactionSlaveAllocation	C ON A.ExposureKey=C.MasterAccount
    --LEFT JOIN TransactionSlaveAllocation    E ON A.MasterAccount=E.MasterAccount
           '''
) 
TEMP5.createOrReplaceTempView("TEMP5") 

In [0]:
%python
TEMP6 = spark.sql(
f'''	
		SELECT A.*,
		CAST(CASE WHEN IFNULL(A.CashTransactions,0)>=IFNULL(A.EoMDelinquentAmount,0) THEN IFNULL(A.EoMDelinquentAmount,0) ELSE IFNULL(A.CashTransactions,0) END AS FLOAT) AS CashTransactionsRestricted,
		CAST(CASE WHEN IFNULL(A.CashPayments,0)>=IFNULL(A.EoMDelinquentAmount,0) THEN IFNULL(A.EoMDelinquentAmount,0) ELSE IFNULL(A.CashPayments,0)		   END  AS FLOAT) AS CashPaymentsRestricted,
		CASE WHEN A.SoMBucketGroup='Current'		AND A.MtDBucketGroup IN('Current','Closed')						THEN A.SoMGBVEur
			 WHEN A.SoMBucketGroup='1'			    AND A.MtDBucketGroup='1'										THEN A.SoMGBVEur
			 WHEN A.SoMBucketGroup='2'			    AND A.MtDBucketGroup='2'										THEN A.SoMGBVEur
			 WHEN A.SoMBucketGroup='3'			    AND A.MtDBucketGroup='3'										THEN A.SoMGBVEur
			 WHEN A.SoMBucketGroup='4+'			    AND A.MtDBucketGroup IN ('4+','Denounced')						THEN A.SoMGBVEur
			 WHEN A.SoMBucketGroup='Denounced'	    AND A.MtDBucketGroup='Denounced'								THEN A.SoMGBVEur
			 ELSE 0 END AS RemainBalance,
		CASE WHEN A.SoMBucketGroup='Current'		AND A.MtDBucketGroup IN ('1','2','3','4+','Denounced')			THEN A.SoMGBVEur 
			   WHEN A.SoMBucketGroup='1'			    AND A.MtDBucketGroup IN ('2','3','4+','Denounced')			   	THEN A.SoMGBVEur 
			   WHEN A.SoMBucketGroup='2'			    AND A.MtDBucketGroup IN ('3','4+','Denounced')					    THEN A.SoMGBVEur 
			   WHEN A.SoMBucketGroup='3'			    AND A.MtDBucketGroup IN ('4+','Denounced')						      THEN A.SoMGBVEur
			   --WHEN SoMBucketGroup='4+'			AND MtDBucketGroup IN ('Denounced')								THEN SoMGBVEur
			   ELSE 0 END AS RollForwardBalance,
		CASE WHEN A.SoMBucketGroup='1'			    AND A.MtDBucketGroup in ('Closed','Current')						      THEN A.SoMGBVEur
			   WHEN A.SoMBucketGroup='2'			    AND A.MtDBucketGroup in ('Closed','Current','1')					    THEN A.SoMGBVEur
			   WHEN A.SoMBucketGroup='3'			    AND A.MtDBucketGroup in ('Closed','Current','1','2')				  THEN A.SoMGBVEur
			   WHEN A.SoMBucketGroup='4+'			    AND A.MtDBucketGroup in ('Closed','Current','1','2','3')			THEN A.SoMGBVEur
			   WHEN A.SoMBucketGroup='Denounced'	AND A.MtDBucketGroup in ('Closed','Current','1','2','3')			THEN A.SoMGBVEur
			   ELSE 0 END AS RollBackBalance,
    CASE WHEN RAA.NewOrAlphaProduct = 'New' AND RAA.GModifiedAccount='G_Modified'  THEN 1
         WHEN RAA.NewOrAlphaProduct <> 'New' AND RAA.GModifiedAccount='G_Modified'  THEN 0
         ELSE NULL END AS NewOrAlphaProduct
        FROM TEMP5                   A
        LEFT  JOIN RA_Assignment     RAA ON a.ExposureKey = RAA.ExposureKey
'''
) 
TEMP6.createOrReplaceTempView("TEMP6") 

In [0]:
%python
TEMP7_0 = spark.sql(
f'''			
    SELECT *,
		CASE WHEN	ReportingCategory = 'Current'									AND 
					IFNULL(IntramonthFlag,0)=0										AND
					GAccountFlag='Modified'
			 THEN	'01_Current_G_Modified'
			 WHEN	ReportingCategory = 'Current'									AND 
					IFNULL(IntramonthFlag,0)=0										AND
					GAccountFlag='NonModified'
			 THEN	'02_Current_Not_G_Modified'
			 WHEN	ReportingCategory = 'Payers_Unhandled'							AND 
					IFNULL(IntramonthFlag,0)=0										AND
					GAccountFlag='Modified'
			 THEN	'03_Unhandled_Payers_G_Modified'
			 WHEN	ReportingCategory = 'Payers_Unhandled'							AND 
					IFNULL(IntramonthFlag,0)=0										AND
					GAccountFlag='NonModified'
			 THEN	'04_Unhandled_Payers_Not_G_Modified'
			 WHEN	ReportingCategory = 'Payers_PJ_Assigned'						AND 
					GAccountFlag='Modified' AND NewOrAlphaProduct =1
			 THEN	'05_PJ_Payers_G_Modified_New'		
    		 
            WHEN	ReportingCategory = 'Payers_PJ_Assigned'						AND 
					GAccountFlag='Modified'
			 THEN	'05_PJ_Payers_G_Modified'		
			 WHEN	ReportingCategory = 'Payers_PJ_Assigned'						AND 
					GAccountFlag='NonModified'
			 THEN	'06_PJ_Payers_Not_G_Modified'	
    		
             WHEN	ReportingCategory = 'Collection'								AND 
					IFNULL(IntramonthFlag,0)=0										AND
					SoMBucketGroup IN ('Current','1')								AND
					GAccountFlag='Modified' AND NewOrAlphaProduct =1
			 THEN	'07_Collection_1_SoM_Assignment_G_Modified_New'
			 WHEN	ReportingCategory = 'Collection'								AND 
					IFNULL(IntramonthFlag,0)=0										AND
					SoMBucketGroup IN ('Current','1')								AND
					GAccountFlag='Modified'
			 THEN	'07_Collection_1_SoM_Assignment_G_Modified'
			 WHEN	ReportingCategory = 'Collection'								AND 
					IFNULL(IntramonthFlag,0)=1										AND
					SoMBucketGroup IN ('Denounced')									AND
					GAccountFlag='Modified' AND NewOrAlphaProduct =1
			 THEN	'07_Collection_1_SoM_Assignment_G_Modified_New'
             WHEN	ReportingCategory = 'Collection'								AND 
					IFNULL(IntramonthFlag,0)=1										AND
					SoMBucketGroup IN ('Denounced')									AND
					GAccountFlag='Modified' 
			 THEN	'07_Collection_1_SoM_Assignment_G_Modified'

			 WHEN	ReportingCategory = 'Collection_Intramonth'						AND 
					IntramonthFlag =1												AND
					GAccountFlag='Modified' AND NewOrAlphaProduct =1
			 THEN	'08_Collection_1_Intramonth_Assignment_G_Modified_New'
			 WHEN	ReportingCategory = 'Collection_Intramonth_2'					AND 
					IntramonthFlag =2												AND
					GAccountFlag='Modified' AND NewOrAlphaProduct =1
			 THEN	'08_Collection_1_Intramonth_Assignment_G_Modified_2_New'
    
    
    		WHEN	ReportingCategory = 'Collection_Intramonth'						AND 
					IntramonthFlag =1												AND
					GAccountFlag='Modified'
			 THEN	'08_Collection_1_Intramonth_Assignment_G_Modified'
			 WHEN	ReportingCategory = 'Collection_Intramonth_2'					AND 
					IntramonthFlag =2												AND
					GAccountFlag='Modified'
			 THEN	'08_Collection_1_Intramonth_Assignment_G_Modified_2'
    
			 WHEN	ReportingCategory = 'Collection'								AND 
					IFNULL(IntramonthFlag,0)=0										AND
					SoMBucketGroup IN ('2')											AND
					GAccountFlag='Modified' AND NewOrAlphaProduct =1
			 THEN	'09_Collection_2_G_Modified_New'
    
             WHEN	ReportingCategory = 'Collection'								AND 
					IFNULL(IntramonthFlag,0)=0										AND
					SoMBucketGroup IN ('2')											AND
					GAccountFlag='Modified'
			 THEN	'09_Collection_2_G_Modified'
			 
             WHEN	ReportingCategory = 'Collection'								AND 
					IFNULL(IntramonthFlag,0)=0										AND
					SoMBucketGroup IN ('3')											AND
					GAccountFlag='Modified' AND NewOrAlphaProduct =1
			 THEN	'10_Collection_3_G_Modified_New'
    
    
            WHEN	ReportingCategory = 'Collection'								AND 
					IFNULL(IntramonthFlag,0)=0										AND
					SoMBucketGroup IN ('3')											AND
					GAccountFlag='Modified'
			 THEN	'10_Collection_3_G_Modified'
			 WHEN	ReportingCategory = 'Collection'								AND 
					IFNULL(IntramonthFlag,0)=0										AND
					SoMBucketGroup IN ('4+')										AND
					GAccountFlag='Modified' AND NewOrAlphaProduct =1
			 THEN	'11_Collection_4+_G_Modified_New'
    
             WHEN	ReportingCategory = 'Collection'								AND 
					IFNULL(IntramonthFlag,0)=0										AND
					SoMBucketGroup IN ('4+')										AND
					GAccountFlag='Modified'
			 THEN	'11_Collection_4+_G_Modified'
			 WHEN	ReportingCategory = 'Collection'								AND 
					IFNULL(IntramonthFlag,0)=0										AND
					SoMBucketGroup IN ('Denounced')									AND
					GAccountFlag='Modified' AND NewOrAlphaProduct =1
			 THEN	'12_Collection_Denounced_G_Modified_New'
    
             WHEN	ReportingCategory = 'Collection'								AND 
					IFNULL(IntramonthFlag,0)=0										AND
					SoMBucketGroup IN ('Denounced')									AND
					GAccountFlag='Modified'
			 THEN	'12_Collection_Denounced_G_Modified'
			 WHEN	ReportingCategory = 'Collection'								AND 
					IFNULL(IntramonthFlag,0)=0										AND
					SoMBucketGroup IN ('Current','1')								AND
					GAccountFlag='NonModified'
			 THEN	'13_Collection_1_SoM_Assignment_Not_G_Modified'
			 WHEN	ReportingCategory = 'Collection'								AND 
					IFNULL(IntramonthFlag,0)=1										AND
					SoMBucketGroup IN ('Denounced')									AND
					GAccountFlag='NonModified'
			 THEN	'13_Collection_1_SoM_Assignment_Not_G_Modified'
			 WHEN	ReportingCategory = 'Collection_Intramonth'						AND 
					IntramonthFlag=1												AND
					GAccountFlag='NonModified'
			 THEN	'14_Collection_1_Intramonth_Assignment_Not_G_Modified'
			 WHEN	ReportingCategory = 'Collection_Intramonth_2'					AND 
					IntramonthFlag=2												AND
					GAccountFlag='NonModified'
			 THEN	'14_Collection_1_Intramonth_Assignment_Not_G_Modified_2'
			 WHEN	ReportingCategory = 'Collection'								AND 
					IFNULL(IntramonthFlag,0)=0										AND
					SoMBucketGroup IN ('2')											AND
					GAccountFlag='NonModified'
			 THEN	'15_Collection_2_Not_G_Modified'
			 WHEN	ReportingCategory = 'Collection'								AND 
					IFNULL(IntramonthFlag,0)=0										AND
					SoMBucketGroup IN ('3')											AND
					GAccountFlag='NonModified'
			 THEN	'16_Collection_3_Not_G_Modified'
			 WHEN	ReportingCategory = 'Collection'								AND 
					IFNULL(IntramonthFlag,0)=0										AND
					SoMBucketGroup IN ('4+')										AND
					GAccountFlag='NonModified'
			 THEN	'17_Collection_4+_Not_G_Modified'
			 WHEN	ReportingCategory = 'Collection'								AND 
					IFNULL(IntramonthFlag,0)=0										AND
					SoMBucketGroup IN ('Denounced')									AND
					GAccountFlag='NonModified'
			 THEN	'18_Collection_Denounced_Not_G_Modified'
			 WHEN	ReportingCategory = 'Non_Payers_Assigned'						AND 
					IFNULL(IntramonthFlag,0)=0
			 THEN	'19_Non_Payers_Assigned'
			 WHEN	ReportingCategory = 'Non_Payers_Not_Assigned'					AND 
					IFNULL(IntramonthFlag,0)=0
			 THEN	'20_Non_Payers_Not_Assigned'
			 WHEN	ReportingCategory = '3869_Positive_Court_Current'				AND 
					IFNULL(IntramonthFlag,0)=0
			 THEN	'21_Positive_Court_Decision_Current'
			 WHEN	ReportingCategory = '3869_Positive_Court_Collection'			AND 
					IFNULL(IntramonthFlag,0)=0										AND
					SoMBucketGroup IN ('Current','1')			 
			 THEN	'22_Positive_Court_Decision_Collection_1_SoM_Assignment'
			 WHEN	ReportingCategory = '3869_Positive_Court_Collection'			AND 
					IFNULL(IntramonthFlag,0)=1										AND
					SoMBucketGroup IN ('Denounced')			 
			 THEN	'22_Positive_Court_Decision_Collection_1_SoM_Assignment'
			 WHEN	ReportingCategory = '3869_Positive_Court_Collection_Intramonth'	AND 
					IntramonthFlag=1
			 THEN	'23_Positive_Court_Decision_Collection_Intramonth_Assignment'
			 WHEN	ReportingCategory = '3869_Positive_Court_Collection_Intramonth_2'	AND 
					IntramonthFlag=2
			 THEN	'23_Positive_Court_Decision_Collection_Intramonth_Assignment_2'
			 WHEN	ReportingCategory = '3869_Positive_Court_Collection'			AND 
					IFNULL(IntramonthFlag,0)=0										AND
					SoMBucketGroup IN ('2')			 
			 THEN	'24_Positive_Court_Decision_Collection_2'
			 WHEN	ReportingCategory = '3869_Positive_Court_Collection'			AND 
					IFNULL(IntramonthFlag,0)=0										AND
					SoMBucketGroup IN ('3')			 
			 THEN	'25_Positive_Court_Decision_Collection_3'
			 WHEN	ReportingCategory = '3869_Positive_Court_Collection'			AND 
					IFNULL(IntramonthFlag,0)=0										AND
					SoMBucketGroup IN ('4+')			 
			 THEN	'26_Positive_Court_Decision_Collection_4+'
			 WHEN	ReportingCategory = '3869_Positive_Court_Collection'			AND 
					IFNULL(IntramonthFlag,0)=0										AND
					SoMBucketGroup IN ('Denounced')			 
			 THEN	'27_Positive_Court_Decision_Collection_Denounced'
			 WHEN	ReportingCategory = '3869_Positive_Court_Unhandled'				AND 
					IFNULL(IntramonthFlag,0)=0
			 THEN	'28_Positive_Court_Decision_Rest_Unhandled'
			 WHEN	ReportingCategory = '3869_Rest'
			 THEN	'29_L3869_Rest'
			 WHEN	ReportingCategory = 'OffBalance'
			 THEN	'30_Off_Balance'
			 ELSE	'31_Check'
			 END AS ReportingCategoryLevel3       
		FROM TEMP6                   
'''
) 
TEMP7_0.createOrReplaceTempView("TEMP7_0")   

In [0]:
%python
TEMP7 = spark.sql(
f'''	
SELECT *,
		CASE WHEN ReportingCategoryLevel3 LIKE '%Current%'																THEN '01_Current_Payers'
		     WHEN ReportingCategoryLevel3 LIKE '%Unhandled%'															THEN '02_Unhandled_Payers'
			 WHEN ReportingCategoryLevel3 LIKE '%PJ%'																	THEN '03_PJ_Payers'
		     WHEN ReportingCategoryLevel3 like '%Collection%G_Modified%' AND ReportingCategoryLevel3 NOT like '%Not%' AND NewOrAlphaProduct =1	THEN '04_Collection_G_Modified_New'
		     WHEN ReportingCategoryLevel3 like '%Collection%G_Modified%' AND ReportingCategoryLevel3 NOT like '%Not%'	THEN '04_Collection_G_Modified'
		     WHEN ReportingCategoryLevel3 like '%Collection%G_Modified%'												THEN '05_Collection_Not_G_Modified'
			 WHEN ReportingCategoryLevel3 like '%Non_Payers%'															THEN '06_Non_Payers'
			 WHEN ReportingCategoryLevel3 like '%Positive_Court%'														THEN '07_L3869_Positive_Court_Decision'
			 WHEN ReportingCategoryLevel3 like '%Rest%'																	THEN '08_L3869_Rest'
			 WHEN ReportingCategoryLevel3 like '%Off%'																	THEN '09_Off_Balance'
		ELSE 'Check'
		END ReportingCategoryLevel2,
		CASE WHEN ReportingCategoryLevel3 LIKE '%Current%'																THEN '01_Payers'
		     WHEN ReportingCategoryLevel3 LIKE '%Unhandled%'															THEN '01_Payers'
			 WHEN ReportingCategoryLevel3 LIKE '%PJ%'																	THEN '01_Payers'
		     WHEN ReportingCategoryLevel3 like '%Collection%G_Modified%' AND ReportingCategoryLevel3 NOT like '%Not%'	THEN '02_Collection_Payers'
		     WHEN ReportingCategoryLevel3 like '%Collection%G_Modified%'												THEN '02_Collection_Payers'
			 WHEN ReportingCategoryLevel3 like '%Non_Payers%'															THEN '03_Non_Payers'
			 WHEN ReportingCategoryLevel3 like '%Positive_Court%'														THEN '04_L3869'
			 WHEN ReportingCategoryLevel3 like '%Rest%'																	THEN '04_L3869'
			 WHEN ReportingCategoryLevel3 like '%Off%'																	THEN '05_Off_Balance'
		ELSE 'Check'
		END ReportingCategoryLevel1		
		FROM TEMP7_0
'''
) 
TEMP7.createOrReplaceTempView("TEMP7")      

In [0]:
%python
AdditionalSegm = spark.sql(
f'''
SELECT AgreementNumber,
	   CASE WHEN ModifiedSegment=1 THEN '01. Payer - G Modified' 
			WHEN PayerSegment=1 THEN '02. Payer - Non G Modified'
			ELSE '04. Non Payer - Non G Modified'
		END AdditionalSegm
FROM (SELECT AgreementNumber,
		     MAX(CASE WHEN CustomerPayerSegment='Payer' THEN 1 ELSE 0 END) PayerSegment ,
		     MAX(CASE WHEN GAccountFlag='Modified' THEN 1 ELSE 0 END) ModifiedSegment
	 FROM TEMP7
	 WHERE ContractLevelAssignmentCategory is null
	   and AccountBalanceFlag='Onbalance'
	 GROUP BY AgreementNumber
	 ) A
'''
) 
AdditionalSegm.createOrReplaceTempView("AdditionalSegm")     

##### Final

In [0]:
%python
CollectionsReport = spark.sql(
f'''
		SELECT 
					a.SoMReferenceDate
			,		a.ReferenceDate
			,		a.ExposureKey			
			,		a.CDI DebtorCDI
			,		a.AgreementNumber
			,		a.Portfolio
			,		a.SoMIsActiveFlag
			,		a.AccountBalanceFlag
			,		a.AccountType
			,		a.GAccountFlag
			,		a.AssetSegment
			,		a.SubProductDescription
			,		a.ProductDescription
			,		a.SoMBucket
			,		a.SoMBucketGroup
			,		a.SoMArrearsStatusEng
			,		a.MtDMBucket
			,		a.MtDBucketGroup
			,		a.ObjectiveStatus
			,		a.ObjectiveDate
			,		a.CustomerPayerSegment
			,		a.AccountPayerSegment
			,		a.FinalSegment
			,		a.OfficialSegment
			,		a.L3869Flag
			,		a.SoML3869Flag
			,		a.SoML3869Status
			,		a.MasterAccount
			,		a.FullyWoff
			,		a.FullyClosed
			,		a.HasActiveSettlementFlag
			,		a.NextInstallmentDate
			,		a.NextInstallmentAmount
			,		a.SoMTotalLegalClaimEur
			,		a.SoMGBVEur
			,		a.onbalanceamount
			,		a.offbalanceamount
			,		a.SoMDelinquent
			,		a.EoMDelinquentAmount
			,		b.TargetRate 
			,		IFNULL(b.TargetRate,0) * EoMDelinquentAmount AS TargetAmount
			,		a.RemainBalance
			,		a.RollForwardBalance
			,		a.RollBackBalance
			,		COALESCE(a.ContractLevelAssignmentCategory,C.AdditionalSegm,'08. OffBalance / Closed') ContractLevelAssignmentCategory
			,		a.ReportingCategoryLevel3
			,		a.ReportingCategoryLevel2
			,		a.ReportingCategoryLevel1
			,		a.CashTransactions
			,		a.LiquidationTransactions
			,		a.CashPayments
			,		a.Liquidations
			,		a.CashTransactionsRestricted
			,		a.CashPaymentsRestricted
			,		a.QueueDescription	
			,		a.AssignmentChannel	
			,		a.AssignmentTeam
			,		a.AssignmentAgent
			,		a.IntramonthFlag
      ,   d.ApplicationImplementationStatus
      ,   a.NewCashPayments AS DownPayment
    --  ,   f.NewDPOPayment AS DPOPayment
      ,   CASE WHEN P.unifiedLoan_AccountNumberWithPrefix = A.ExposureKey THEN a.CashPayments
               ELSE 0 END AS DPOPayment
      ,   g.ExpenseRequestAmount
      ,   a.HighBalanceFlag  
	FROM TEMP7 A
	LEFT JOIN OperationalTargets B ON A.ReportingCategoryLevel3=B.MatchingCategory and b.ReferenceDateEoM='{som_date}' and a.Portfolio=b.Portfolio
	LEFT JOIN AdditionalSegm C ON A.AgreementNumber=C.AgreementNumber
  LEFT JOIN fldt_allapplications     D ON D.unifiedLoan_AccountNumberWithPrefix=A.ExposureKey
  --LEFT JOIN AllocationPayments_Step0 E ON E.PreviousAccountNumber=A.ExposureKey --step5?
  
  --LEFT JOIN DPO_Allocation_Step0     F ON F.PreviousAccountNumber=A.ExposureKey
  LEFT JOIN paymentscheduleExpenses  G ON G.ExposureKey = A.ExposureKey
  LEFT JOIN DPO                      P ON P.unifiedLoan_AccountNumberWithPrefix = A.ExposureKey
'''
) 
CollectionsReport.createOrReplaceTempView("CollectionsReport")    

##### final_df

In [0]:
%python
final_df = spark.sql(
f'''
		SELECT 
					CAST(SoMReferenceDate					   AS DATE)          AS SoMBusinessDate				
			,		CAST(ReferenceDate						   AS DATE)          AS BusinessDate				
			,		CAST(ExposureKey						   AS STRING)        AS ExposureKey					
			,		CAST(DebtorCDI							   AS STRING)        AS CDI						
			,		CAST(AgreementNumber					   AS STRING)        AS AgreementKey				
			,		CAST(Portfolio							   AS STRING)        AS PortfolioName						
			,		CAST(SoMIsActiveFlag					   AS INT)           AS SoMIsActiveFlag				
			,		CAST(AccountBalanceFlag					   AS STRING)        AS AccountBalanceFlag				
			,		CAST(AccountType						   AS STRING)        AS AccountType					
			,		CAST(GAccountFlag						   AS STRING)        AS GAccountFlag					
			,		CAST(AssetSegment						   AS STRING)        AS AssetSegment					
			,		CAST(SubProductDescription				   AS STRING)        AS SubProductDescription			
			,		CAST(ProductDescription					   AS STRING)        AS ProductDescription				
			,		CAST(SoMBucket							   AS INT)           AS SoMBucket						
			,		CAST(SoMBucketGroup						   AS STRING)        AS SoMBucketGroup					
			,		CAST(SoMArrearsStatusEng				   AS STRING)        AS SoMArrearsStatusEng			
			,		CAST(MtDMBucket							   AS INT)           AS MtDMBucket						
			,		CAST(MtDBucketGroup						   AS STRING)        AS MtDBucketGroup					
			,		CAST(ObjectiveStatus					   AS STRING)        AS ObjectiveStatus				
			,		CAST(ObjectiveDate						   AS DATE)          AS ObjectiveDate					
			,		CAST(CustomerPayerSegment				   AS STRING)        AS CustomerPayerSegment			
			,		CAST(AccountPayerSegment				   AS STRING)        AS AccountPayerSegment			
			,		CAST(FinalSegment						   AS STRING)        AS FinalSegment					
			,		CAST(OfficialSegment					   AS STRING)        AS OfficialSegment				
			,		CAST(L3869Flag							   AS STRING)        AS L3869Flag						
			,		CAST(SoML3869Flag						   AS STRING)        AS SoML3869Flag					
			,		CAST(SoML3869Status						   AS STRING)        AS SoML3869Status					
			,		CAST(MasterAccount						   AS STRING)        AS MasterAccount					
			,		CAST(FullyWoff							   AS INT)           AS FullyWoff						
			,		CAST(FullyClosed						   AS INT)           AS FullyClosed					
			,		CAST(HasActiveSettlementFlag			   AS INT)           AS HasActiveSettlementFlag		
			,		CAST(NextInstallmentDate				   AS DATE)          AS NextInstallmentDate			
			,		CAST(NextInstallmentAmount				   AS DECIMAL(19,2)) AS NextInstallmentAmount			
			,		CAST(SoMTotalLegalClaimEur				   AS DECIMAL(19,2)) AS SoMTotalLegalClaimEur			
			,		CAST(SoMGBVEur							   AS DECIMAL(19,2)) AS SoMGBVEur						
			,		CAST(onbalanceamount					   AS DECIMAL(19,2)) AS onbalanceamount				
			,		CAST(offbalanceamount					   AS DECIMAL(19,2)) AS offbalanceamount				
			,		CAST(SoMDelinquent						   AS DECIMAL(19,2)) AS SoMDelinquent					
			,		CAST(EoMDelinquentAmount				   AS DECIMAL(19,2)) AS EoMDelinquentAmount			
			,		CAST(TargetRate 						   AS DECIMAL(19,2)) AS TargetRate 					
			,		CAST(TargetAmount						   AS DECIMAL(19,2)) AS TargetAmount					
			,		CAST(RemainBalance						   AS DECIMAL(19,2)) AS RemainBalance					
			,		CAST(RollForwardBalance					   AS DECIMAL(19,2)) AS RollForwardBalance				
			,		CAST(RollBackBalance					   AS DECIMAL(19,2)) AS RollBackBalance				
			,		CAST(ContractLevelAssignmentCategory	   AS STRING)        AS ContractLevelAssignmentCategory
			,		CAST(ReportingCategoryLevel3			   AS STRING)        AS ReportingCategoryLevel3		
			,		CAST(ReportingCategoryLevel2			   AS STRING)        AS ReportingCategoryLevel2		
			,		CAST(ReportingCategoryLevel1			   AS STRING)        AS ReportingCategoryLevel1		
			,		CAST(CashTransactions					   AS DECIMAL(19,2)) AS CashTransactions				
			,		CAST(LiquidationTransactions			   AS DECIMAL(19,2)) AS LiquidationTransactions		
			,		CAST(CashPayments						   AS DECIMAL(19,2)) AS CashPayments					
			,		CAST(Liquidations						   AS DECIMAL(19,2)) AS Liquidations					
			,		CAST(CashTransactionsRestricted			   AS DECIMAL(19,2)) AS CashTransactionsRestricted		
			,		CAST(CashPaymentsRestricted				   AS DECIMAL(19,2)) AS CashPaymentsRestricted			
			,		CAST(QueueDescription					   AS STRING)        AS QueueDescription				
			,		CAST(AssignmentChannel					   AS STRING)        AS AssignmentChannel				
			,		CAST(AssignmentTeam						   AS STRING)        AS AssignmentTeam					
			,		CAST(AssignmentAgent					   AS STRING)        AS AssignmentAgent				
			,		CAST(IntramonthFlag						   AS INT)           AS IntramonthFlag	
      		,   	CAST(ApplicationImplementationStatus       AS STRING)	     AS ApplicationImplementationStatus
      		,   	CAST(0                                     AS STRING)	     AS Exceptions
      		,   	CAST(DownPayment                           AS DECIMAL(19,2) )AS DownPayment
      		,   	CAST(DPOPayment                            AS DECIMAL(19,2) )AS DPOPayment
      		,  	 	CAST(ExpenseRequestAmount                  AS DECIMAL(19,2) )AS ExpenseRequestAmount
      		,   	CAST(NULL                                  AS DATE)          AS PaymentJumpDate
      		,   	CAST(NULL                                  AS DECIMAL(19,2) )AS PaymentJumpAmount
     		,   	CAST(HighBalanceFlag                       AS STRING)        AS HighBalanceFlag
      		, 		CAST(getdate() 							   AS TIMESTAMP)     AS InsertTimeStamp
		FROM CollectionsReport
'''
) 
final_df.createOrReplaceTempView("final_df")  

In [0]:
%python
final_df.write \
    .format("delta") \
    .partitionBy("BusinessDate") \
    .mode("append") \
    .saveAsTable(delivery_table_path) # in order to write in delta_lake

In [0]:
%python
from pyspark.sql.functions import col, count, concat

def check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="error"):
    errors = []
    warnings = []

    if key_columns:
        # Create the composite key column if there are multiple key columns
        if len(key_columns) > 1:
            composite_key = concat(*[col(c) for c in key_columns]).alias("CompositeKey")
            key_column_name = "CompositeKey"
            key_columns_string = ", ".join(key_columns)
        else:
            key_column_name = key_columns[0]
            key_columns_string = key_columns[0]

        # Duplicate Checks
        duplicate_count = (
            spark.table(delivery_table_path)
            .filter(col("BusinessDate") == run_date)
            .withColumn("CompositeKey", composite_key if len(key_columns) > 1 else col(key_column_name))
            .groupBy(key_column_name)
            .count()
            .filter("count > 1")
            .select(key_column_name, "count")
            .toPandas()
            .to_dict(orient='records')
        )

        if duplicate_count:
            message = f"Duplicate {key_columns_string} entries found: {duplicate_count}"
            if duplicate_check_action == "error":
                errors.append(message)
            elif duplicate_check_action == "warning":
                warnings.append(message)

    # Check if the table for the specific date is empty
    row_count = spark.sql(f"""
        SELECT * FROM {delivery_table_path}
        WHERE BusinessDate = '{run_date}'
    """).count()

    if row_count == 0:
        message = f"The {delivery_table_path} dataset is empty for the date {run_date}."
        if empty_check_action == "error":
            errors.append(message)
        elif empty_check_action == "warning":
            warnings.append(message)

    if warnings:
        print("Warnings:\n" + "\n".join(warnings))

    if errors:
        raise ValueError("Errors found:\n" + "\n".join(errors))

key_columns = ["ExposureKey"]
check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="error")

In [0]:
%python
import re
def rename_and_cleanup_files(output_path, file_name):
    part_file_regex = re.compile(r'part-.*')
    system_file_regex = re.compile(r'^_(committed_|started_|SUCCESS)')
 
    # Move the part file to the final filename.
    files = dbutils.fs.ls(output_path)
    for file in files:
        print(f"file: {file.name}")
        if part_file_regex.match(file.name):
            original_path = file.path
            new_path = output_path + file_name
            dbutils.fs.mv(original_path, new_path)
 
    # Remove system files from the common folder.
    files = dbutils.fs.ls(output_path)
    for file in files:
        if system_file_regex.match(file.name):
            dbutils.fs.rm(file.path, recurse=False)
 
# Example usage:
# rename_and_cleanup_files(output_path, file_name)

In [0]:
%python
CollectionsReport = spark.sql(
f'''
select * from {delivery_table_path}
where BusinessDate = {run_date!r}
'''
)
CollectionsReport.createOrReplaceTempView("CollectionsReport")

In [0]:
%python
output_path = "/Volumes/"+env_prefix+"-cdh-delivery/repoint/exports/HAPS/CollectionsReport/" # 
file_name = "CollectionsReport_" +run_date+ ".csv"
CollectionsReport.coalesce(1).write.mode("append").option("header", True).csv(f"{output_path}") # 
rename_and_cleanup_files(output_path, file_name)